# Beer Game — Role-Wise Parameter Inference

Refactored notebook for the BayesFlow-based parameter inference pipeline.
Same outputs as the original `Beer_Role_Infer.ipynb`, organized into clearly
scoped cells: setup → simulator → BayesFlow → per-experiment analysis →
overlays → PCA / distance plots.


## 1. Configure Data and Result Paths

Input files are read from `/code/beer data/...`, the trained model is loaded from `/code/model_role_cf.keras`, and generated figures and tables are written to `/results/...`.


In [ ]:
from pathlib import Path

# Input files are expected under:
#   /code/beer data/<model-folder>/<file>
# Trained model:
#   /code/model_role_cf.keras
# Generated figures and CSV tables are written under:
#   /results/<filename>
DATA_ROOT = Path("/code/beer data")
RESULTS_ROOT = Path("/results")
MODEL_PATH = Path("/code/model_role_cf.keras")
PLOT_DIR = RESULTS_ROOT

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


## 2. Install Dependencies


In [ ]:
!pip install -U "numpy==1.26.4" "bayesflow==2.0.7" "keras==3.10.0" "jax==0.6.0" "jaxlib==0.6.0"

## 3. Imports, Backend, and Global Constants

Keras backend is set **before** importing keras. We also fix a global random
seed so every downstream sample/train call is reproducible.


In [ ]:
import os
if "KERAS_BACKEND" not in os.environ:
    os.environ["KERAS_BACKEND"] = "jax"   # alternatives: "torch", "tensorflow"

import datetime, math, random, json, itertools
from glob import glob
from pathlib import Path
from collections import deque
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
from matplotlib.colors import to_rgb
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns

from scipy.stats import ttest_ind, pearsonr, spearmanr, zscore
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

import keras
from numba import jit
import bayesflow as bf

# ----- Reproducibility -----
SEED = 42
keras.utils.set_random_seed(SEED)

# ----- Supply-chain roles (used everywhere) -----
ROLES = ["retailer", "wholesaler", "distributor", "factory"]

### Figure style

Shared style and helpers for the paper figures: 7-pt sans-serif text, vector PDF with editable text, keys inside each figure, and individual sessions drawn as dots over the bars. Each figure keeps its original colours; colours that failed a colour-blindness check were replaced by the nearest passing colour. The style is applied only inside the figure functions, so other plots (e.g. the BayesFlow diagnostics) keep their defaults.

In [ ]:
# Figure style for the paper figures (Nature format): 7-pt sans-serif text, vector PDF with
# editable text, keys inside the figure, and individual sessions shown as dots aligned in
# their own column. The style is applied only inside the figure functions (via @nature_style),
# so other plots in this notebook (e.g. the BayesFlow diagnostics) keep their defaults.
# Colours: each figure keeps its original colours; colours that failed a colour-blindness check
# (every pair at least as distinguishable as the Okabe-Ito palette under simulated deuteranopia,
# protanopia and tritanopia) were replaced by the nearest passing colour.
import matplotlib as mpl
import matplotlib.font_manager
from functools import wraps
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

MM = 1 / 25.4
FULL_WIDTH, HALF_WIDTH = 180 * MM, 88 * MM
# Bar charts: Set2 resampled to 7 colours (as in the original plots), repaired.
SET2_BARS = ["#50a08a", "#fc8d62", "#8da0cb", "#b6ff8a", "#ffd92f", "#d3d3af", "#b3b3b3"]
# Radar charts: the first 7 listed Set2 colours, repaired.
SET2_CVD = ["#66c2a5", "#fc8d62", "#8da0cb", "#c575a0", "#b6ff8a", "#ffd92f", "#dbdbbd"]
mpl.colormaps.register(mpl.colors.ListedColormap(SET2_CVD, name="Set2_cvd"), force=True)
# PCA scatter: tab10 assignments of the original plots, repaired.
PCA_COLORS = {"GPT-OSS 120B": "#33668a", "GPT-OSS 20B": "#f86d00", "Kimi K2": "#6d9957",
              "Llama-3.1 8B": "#d3241d", "Llama-3.3 70B": "#9467bd", "Mistral Small": "#8c564b",
              "Qwen-3.6-35B-A3B": "#e377c2"}
BENCHMARK_X = "#16a085"
CR_COLORS = {"low CR": "#c51d7c", "high CR": "#d95f02"}
PARAM_LABELS = {"theta": r"$\theta$", "alpha_s": r"$\alpha_S$", "beta": r"$\beta$", "s_prime": r"$S$′"}
ROLE_OF = {"r": "Retailer", "w": "Wholesaler", "d": "Distributor", "f": "Factory"}
DOT_COLOR = "#222222"
HATCH = "////"
DOT_SEP = {"linear": 0.02, "log": 0.02}   # set per figure by set_dot_spacing()

# Arial where installed (Nature's preferred font); otherwise the closest metric-compatible sans.
_INSTALLED = {f.name for f in mpl.font_manager.fontManager.ttflist}
SANS = next((f for f in ["Arial", "Helvetica", "Liberation Sans"] if f in _INSTALLED), "DejaVu Sans")
NATURE_RC = {
    "font.family": "sans-serif", "font.sans-serif": [SANS, "DejaVu Sans"],
    "mathtext.fontset": "custom", "mathtext.rm": SANS, "mathtext.it": f"{SANS}:italic",
    "mathtext.bf": f"{SANS}:bold", "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6, "legend.title_fontsize": 6,
    "axes.linewidth": 0.5, "xtick.major.width": 0.5, "ytick.major.width": 0.5,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "lines.linewidth": 1.0, "patch.linewidth": 0.5,
    "hatch.linewidth": 0.4, "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "legend.frameon": False, "savefig.dpi": 600,
}


def nature_style(func):
    """Draw (and save) the figure under the paper's figure style."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        with mpl.rc_context(NATURE_RC):
            return func(*args, **kwargs)
    return wrapper


def save_figure(fig, savepath):
    if savepath:
        fig.savefig(savepath, bbox_inches="tight", pad_inches=0.02)


def lighten(color, amount=0.45):
    r, g, b = mpl.colors.to_rgb(color)
    return (1 - amount * (1 - r), 1 - amount * (1 - g), 1 - amount * (1 - b))


def star(p):
    if not np.isfinite(p):
        return ""
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""


def sem(v):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    return v.std(ddof=1) / np.sqrt(v.size) if v.size > 1 else np.nan


def set_dot_spacing(all_values, log=False, frac=0.012):
    """Vertical distance below which two dots in the same column are shifted sideways."""
    v = np.asarray([x for x in np.ravel(all_values) if np.isfinite(x) and (x > 0 or not log)], float)
    v = np.log10(v) if log else v
    DOT_SEP["log" if log else "linear"] = max((v.max() - v.min()) * frac, 1e-9)


def swarm_offsets(values, y_sep, x_step, max_half):
    """Deterministic beeswarm: each dot sits on the column centre unless it would overlap another."""
    order = np.argsort(values)
    placed, offs = [], np.zeros(len(values))
    for idx in order:
        y = values[idx]
        cand = 0.0
        for k in range(1, 60):
            if all(abs(y - py) >= y_sep or abs(cand - px) >= x_step * 0.999 for px, py in placed):
                break
            cand = ((k + 1) // 2) * x_step * (1 if k % 2 else -1)
            if abs(cand) > max_half:
                cand = float(np.clip(cand, -max_half, max_half))
                break
        placed.append((cand, y))
        offs[idx] = cand
    return offs


def dots(ax, x, values, width, size=1.0, log=False):
    values = np.asarray(values, float)
    values = values[np.isfinite(values) & ((values > 0) if log else True)]
    yv = np.log10(values) if log else values
    offs = swarm_offsets(yv, DOT_SEP["log" if log else "linear"], width * 0.2, width * 0.42)
    ax.scatter(x + offs, values, s=size, color=DOT_COLOR, alpha=0.75, linewidths=0, zorder=4, clip_on=False)


def bar_with_dots(ax, x, values, color, width, baseline=False, err="sem", center=None, dot_size=3, show_dots=True):
    """Bar at the mean (or a supplied centre) with an error bar and the individual values as dots.

    err: "sem" (s.e.m. of `values`), None (no error bar) or an explicit (lower, upper) pair.
    Baseline bars are lighter and hatched.
    """
    values = np.asarray(values, float)
    if center is None:
        center = np.nanmean(values)
    ax.bar(x, center, width=width, color=lighten(color) if baseline else color,
           edgecolor="#555555" if baseline else "black", linewidth=0.4,
           hatch=HATCH if baseline else None, zorder=2)
    if err == "sem":
        e = sem(values)
        ax.errorbar(x, center, yerr=e, fmt="none", ecolor="black", elinewidth=0.5, capsize=1.2, zorder=5)
    elif err is not None:
        lo, hi = err
        ax.errorbar(x, center, yerr=[[center - lo], [hi - center]], fmt="none", ecolor="black",
                    elinewidth=0.5, capsize=1.2, zorder=5)
    if show_dots:
        dots(ax, x, values, width, size=dot_size, log=(ax.get_yscale() == "log"))


def condition_handles(extra=()):
    return [Patch(facecolor="white", edgecolor="#555555", hatch=HATCH, label="Baseline"),
            Patch(facecolor="#999999", edgecolor="black", label="Heuristic-aware"),
            Line2D([], [], marker="o", ls="", color=DOT_COLOR, alpha=0.75, markeredgewidth=0, markersize=2.5,
                   label="Session ($n=11$)"), *extra]

## 4. Beer Game Simulator (Mass-Conserving ABM)

The core agent-based simulator. Returns a dict with `data` (weekly DataFrame),
`total_costs`, `amplification`, parameter snapshot, and a sanity summary.


In [ ]:
def simulate_beer_game(
    # Retailer
    theta_r=0.36, alpha_s_r=0.26, beta_r=0.34, s_prime_r=17,
    # Wholesaler
    theta_w=0.36, alpha_s_w=0.26, beta_w=0.34, s_prime_w=17,
    # Distributor
    theta_d=0.36, alpha_s_d=0.26, beta_d=0.34, s_prime_d=17,
    # Factory
    theta_f=0.36, alpha_s_f=0.26, beta_f=0.34, s_prime_f=17,
    # Noise and seed
    sigma_eps=1,
    seed=42,
    # Environment settings aligned with the latest LLM script
    weeks=35,
    initial_inventory=12,
    initial_throughput=4,
    # Backward-compatible alias for old calling code
    initial_order=None,
    order_delay=2,
    shipment_lead_time=2,
    production_lead_time=2,
    forced_warmup_rounds=4,
    run_sanity_check=True,
):
    """
    Mass-conserving ABM Beer Game aligned with the latest LLM simulator.

    Aligned game mechanics:
    - 35 simulated rounds by default.
    - Customer demand is 4 for the first 4 rounds, then 8.
    - 2-round order delay.
    - 2-round shipment delay.
    - 2-round factory production lead time.
    - Initial Sterman-style equilibrium:
        * each role starts with 12 units of physical inventory
        * throughput is initially 4 units per round
        * order-delay pipelines contain 4-unit order slips
        * shipment/production pipelines contain 4 physical units
    - First 4 rounds are forced warm-up rounds:
        * all roles order/produce exactly 4
        * ABM decision rule is not used for those orders
    - Inventory is physical on-hand stock only and is never negative.
    - Backlog is tracked separately.
    - Downstream deliveries are scheduled from actual shipped quantities,
      not from order quantities.
    - Backlog fulfillment is physically delivered downstream.
    - outstanding_to_supplier tracks orders/production not yet physically received.

    ABM behavioral rule after warm-up:

        IO_t = L_e + alpha_s * (S' - S_t - beta * SL_t) + epsilon_t

    where:
    - L_e is adaptive expected loss/demand.
    - S_t is effective inventory = on-hand inventory - backlog.
    - SL_t is outstanding_to_supplier before placing the current order.
    - epsilon_t is Gaussian noise with standard deviation sigma_eps.
    """

    import pandas as pd
    import numpy as np
    # ---- Backward compatibility ----
    # Old code may call simulate_beer_game(initial_order=4).
    # The latest LLM script calls this concept INITIAL_THROUGHPUT.
    if initial_order is not None:
        initial_throughput = initial_order
    # ---- Validation ----

    if weeks < 1:
        raise ValueError("weeks must be at least 1.")
    if order_delay < 1:
        raise ValueError("order_delay must be at least 1.")
    if shipment_lead_time < 1:
        raise ValueError("shipment_lead_time must be at least 1.")
    if production_lead_time < 1:
        raise ValueError("production_lead_time must be at least 1.")
    if forced_warmup_rounds < 0:
        raise ValueError("forced_warmup_rounds must be nonnegative.")

    rng = np.random.default_rng(seed)

    roles = ["retailer", "wholesaler", "distributor", "factory"]

    params = {
        "retailer": {
            "theta": theta_r,
            "alpha_s": alpha_s_r,
            "beta": beta_r,
            "s_prime": s_prime_r,
        },
        "wholesaler": {
            "theta": theta_w,
            "alpha_s": alpha_s_w,
            "beta": beta_w,
            "s_prime": s_prime_w,
        },
        "distributor": {
            "theta": theta_d,
            "alpha_s": alpha_s_d,
            "beta": beta_d,
            "s_prime": s_prime_d,
        },
        "factory": {
            "theta": theta_f,
            "alpha_s": alpha_s_f,
            "beta": beta_f,
            "s_prime": s_prime_f,
        },
    }

    # Box constraints for stability.
    for p in params.values():
        p["theta"] = min(1.0, max(0.0, p["theta"]))
        p["alpha_s"] = max(0.0, p["alpha_s"])
        p["beta"] = min(1.0, max(0.0, p["beta"]))
        p["s_prime"] = max(0.0, p["s_prime"])

    # ---- Utility helpers ----

    def to_nonnegative_int(value, default=0):
        try:
            value = int(round(float(value)))
        except (TypeError, ValueError):
            return default
        return max(0, value)

    initial_inventory = to_nonnegative_int(initial_inventory)
    initial_throughput = to_nonnegative_int(initial_throughput)

    customer_demand = [
        initial_throughput if week < forced_warmup_rounds else 8
        for week in range(weeks)
    ]

    def make_seeded_delay_pipeline(delay_length, qty=initial_throughput):
        """
        Create a seeded delay pipeline matching the LLM script.

        Because pipelines are advanced at the beginning of each round:
        - eta = 1 arrives in the current round after advance.
        - eta = 2 arrives in the next round after advance.
        """
        return [
            {"qty": to_nonnegative_int(qty), "eta": eta}
            for eta in range(1, int(delay_length) + 1)
        ]

    def advance_pipeline(pipeline):
        for item in pipeline:
            item["eta"] -= 1

    def pop_arrivals(pipeline):
        arriving = [item for item in pipeline if item["eta"] <= 0]
        remaining = [item for item in pipeline if item["eta"] > 0]

        pipeline.clear()
        pipeline.extend(remaining)

        return sum(to_nonnegative_int(item.get("qty", 0)) for item in arriving)

    def add_to_pipeline(pipeline, qty, eta):
        qty = to_nonnegative_int(qty)
        eta = int(eta)

        if qty > 0:
            pipeline.append({"qty": qty, "eta": eta})

    def pipeline_sum(pipeline):
        return sum(to_nonnegative_int(item.get("qty", 0)) for item in pipeline)

    def eta_breakdown(pipeline, max_eta=4):
        return {
            f"eta_{eta}": sum(
                to_nonnegative_int(item.get("qty", 0))
                for item in pipeline
                if int(item.get("eta", 0)) == eta
            )
            for eta in range(1, max_eta + 1)
        }

    # ---- Agent initialization: seeded equilibrium ----

    agents = {}

    for role in roles:
        agents[role] = {
            # Physical inventory only.
            "inventory": initial_inventory,

            # Explicit unfilled downstream orders.
            "backlog": 0,

            # Downstream orders that have not yet arrived to this role.
            # Retailer has no meaningful downstream order pipeline because
            # customer demand is exogenous.
            "order_pipeline": [],

            # Inbound physical shipments from supplier.
            # For factory, this is production-in-progress.
            "shipment_pipeline": [],

            # Reporting/bookkeeping.
            "last_order": 0,
            "orders_placed": [],

            # Adaptive-expectation state.
            "expected_loss": float(initial_throughput),
            "orders_history": [initial_throughput] * 4,

            # Holding + backlog costs.
            "total_cost": 0.0,

            # Orders/production not yet physically received.
            "outstanding_to_supplier": 0,
        }

    # Seed order-delay pipelines:
    # Retailer receives customer demand directly.
    # Wholesaler receives Retailer order slips.
    # Distributor receives Wholesaler order slips.
    # Factory receives Distributor order slips.
    for role in ["wholesaler", "distributor", "factory"]:
        agents[role]["order_pipeline"] = make_seeded_delay_pipeline(
            order_delay,
            initial_throughput,
        )

    # Seed physical shipment pipelines:
    # Retailer receives from Wholesaler.
    # Wholesaler receives from Distributor.
    # Distributor receives from Factory.
    for role in ["retailer", "wholesaler", "distributor"]:
        agents[role]["shipment_pipeline"] = make_seeded_delay_pipeline(
            shipment_lead_time,
            initial_throughput,
        )

        # Match the latest LLM script:
        # For non-factory roles, outstanding includes both order-delay boxes
        # and shipment-delay boxes at initialization.
        agents[role]["outstanding_to_supplier"] = (
            order_delay + shipment_lead_time
        ) * initial_throughput

    # Factory has production already in progress.
    agents["factory"]["shipment_pipeline"] = make_seeded_delay_pipeline(
        production_lead_time,
        initial_throughput,
    )

    # Match the latest LLM script:
    # Factory outstanding is production requested but not yet completed.
    agents["factory"]["outstanding_to_supplier"] = (
        production_lead_time * initial_throughput
    )

    # ---- Model helpers ----

    def calc_expected(agent, theta):
        last_actual = (
            agent["orders_history"][-1]
            if agent["orders_history"]
            else float(initial_throughput)
        )
        prev_exp = agent["expected_loss"]

        return max(0.0, theta * last_actual + (1.0 - theta) * prev_exp)

    def append_observed_incoming(agent, incoming):
        """
        Update observed demand/order history.

        This is done in both forced warm-up and ABM-decision rounds so that
        post-warm-up ABM expectations have seen the same incoming-order stream.
        """
        agent["orders_history"].append(to_nonnegative_int(incoming))

        # Keep the same max history length as the older ABM.
        if len(agent["orders_history"]) > 8:
            agent["orders_history"] = agent["orders_history"][-8:]

    def receive(agent, qty):
        """
        Receive actual physical units.

        This matches the LLM accounting convention:
        outstanding_to_supplier decreases only when units are physically received.
        """
        qty = to_nonnegative_int(qty)

        if qty <= 0:
            return

        agent["outstanding_to_supplier"] = max(
            0,
            agent["outstanding_to_supplier"] - qty,
        )
        agent["inventory"] += qty

    def fill(agent, demand_qty):
        """
        Mass-conserving fulfillment.

        1. Add incoming demand/order to backlog.
        2. Ship min(on-hand inventory, backlog).
        3. Reduce inventory only by actual shipped units.
        4. Reduce backlog only by actual shipped units.
        5. Return actual shipped units.

        This ensures old backlog can be fulfilled and physically delivered
        downstream when inventory becomes available.
        """
        demand_qty = to_nonnegative_int(demand_qty)

        agent["backlog"] += demand_qty

        shipped = min(agent["inventory"], agent["backlog"])

        agent["inventory"] -= shipped
        agent["backlog"] -= shipped

        return shipped

    def cost_breakdown(agent):
        inventory_cost = 0.5 * agent["inventory"]
        backlog_cost = 1.0 * agent["backlog"]

        total_cost = inventory_cost + backlog_cost
        agent["total_cost"] += total_cost

        return total_cost, inventory_cost, backlog_cost

    def calc_order(agent, incoming, p):
        """
        ABM ordering rule used only after forced warm-up rounds.

        Uses outstanding_to_supplier before adding the current order, matching
        the LLM prompt state.
        """
        incoming = to_nonnegative_int(incoming)

        agent["expected_loss"] = calc_expected(agent, p["theta"])

        effective_inventory = agent["inventory"] - agent["backlog"]
        SL_t = agent["outstanding_to_supplier"]

        intended_order = agent["expected_loss"] + p["alpha_s"] * (
            p["s_prime"] - effective_inventory - p["beta"] * SL_t
        )

        if sigma_eps > 0:
            intended_order += rng.normal(0.0, sigma_eps)

        order = max(0, int(round(intended_order)))

        append_observed_incoming(agent, incoming)
        agent["orders_placed"].append(order)

        return order

    # ---- Simulation loop ----

    weekly_data = []

    for week in range(weeks):
        round_number = week + 1
        demand = customer_demand[week]

        # 1. Advance order and shipment/production pipelines.
        for role in roles:
            advance_pipeline(agents[role]["order_pipeline"])
            advance_pipeline(agents[role]["shipment_pipeline"])

        # 2. Collect arriving downstream orders.
        order_received = {
            "retailer": demand,
            "wholesaler": pop_arrivals(agents["wholesaler"]["order_pipeline"]),
            "distributor": pop_arrivals(agents["distributor"]["order_pipeline"]),
            "factory": pop_arrivals(agents["factory"]["order_pipeline"]),
        }

        # Retailer has no meaningful downstream order pipeline.
        _ = pop_arrivals(agents["retailer"]["order_pipeline"])

        # 3. Collect arriving physical shipments/production.
        shipment_received = {
            "retailer": pop_arrivals(agents["retailer"]["shipment_pipeline"]),
            "wholesaler": pop_arrivals(agents["wholesaler"]["shipment_pipeline"]),
            "distributor": pop_arrivals(agents["distributor"]["shipment_pipeline"]),
            "factory": pop_arrivals(agents["factory"]["shipment_pipeline"]),
        }

        # 4. Receive physical units.
        for role in roles:
            receive(agents[role], shipment_received[role])

        # 5. Fulfill downstream demand/orders using physical inventory.
        shipped = {
            "retailer": fill(
                agents["retailer"],
                order_received["retailer"],
            ),
            "wholesaler": fill(
                agents["wholesaler"],
                order_received["wholesaler"],
            ),
            "distributor": fill(
                agents["distributor"],
                order_received["distributor"],
            ),
            "factory": fill(
                agents["factory"],
                order_received["factory"],
            ),
        }

        # State visible to the decision rule before current order is added.
        SL_before_order = {
            role: agents[role]["outstanding_to_supplier"]
            for role in roles
        }

        visible_pipeline_before_order = {
            role: pipeline_sum(agents[role]["shipment_pipeline"])
            for role in roles
        }

        visible_eta_before_order = {
            role: eta_breakdown(agents[role]["shipment_pipeline"], max_eta=4)
            for role in roles
        }

        # 6. Place orders.
        # First forced_warmup_rounds rounds are not behavioral ABM decisions.
        orders = {}
        decision_source = {}

        for role in roles:
            if week < forced_warmup_rounds:
                order = initial_throughput
                append_observed_incoming(agents[role], order_received[role])
                agents[role]["orders_placed"].append(order)
                source = "forced_warmup"
            else:
                order = calc_order(
                    agents[role],
                    order_received[role],
                    params[role],
                )
                source = "abm"

            orders[role] = to_nonnegative_int(order)
            decision_source[role] = source

            agents[role]["last_order"] = orders[role]

            # Match latest LLM script:
            # outstanding_to_supplier increases immediately after placing order.
            agents[role]["outstanding_to_supplier"] += orders[role]

        # 7. Schedule new orders into upstream order pipelines.
        # These are information/order flows, not physical shipments.
        add_to_pipeline(
            agents["wholesaler"]["order_pipeline"],
            orders["retailer"],
            order_delay,
        )
        add_to_pipeline(
            agents["distributor"]["order_pipeline"],
            orders["wholesaler"],
            order_delay,
        )
        add_to_pipeline(
            agents["factory"]["order_pipeline"],
            orders["distributor"],
            order_delay,
        )

        # 8. Schedule actual physical shipments downstream.
        # Key mass-conserving rule:
        # downstream receives actual shipped quantities, not order quantities.
        add_to_pipeline(
            agents["retailer"]["shipment_pipeline"],
            shipped["wholesaler"],
            shipment_lead_time,
        )
        add_to_pipeline(
            agents["wholesaler"]["shipment_pipeline"],
            shipped["distributor"],
            shipment_lead_time,
        )
        add_to_pipeline(
            agents["distributor"]["shipment_pipeline"],
            shipped["factory"],
            shipment_lead_time,
        )

        # 9. Factory production creates new physical units after production lead time.
        add_to_pipeline(
            agents["factory"]["shipment_pipeline"],
            orders["factory"],
            production_lead_time,
        )

        # 10. Costs after fulfillment and ordering.
        costs = {}
        for role in roles:
            costs[role] = cost_breakdown(agents[role])

        # 11. Weekly record.
        row = {
            "week": week,
            "round": round_number,
            "customer_demand": demand,

            # Orders placed this round.
            "retailer_order": orders["retailer"],
            "wholesaler_order": orders["wholesaler"],
            "distributor_order": orders["distributor"],
            "factory_order": orders["factory"],

            # Decision source.
            "retailer_decision_source": decision_source["retailer"],
            "wholesaler_decision_source": decision_source["wholesaler"],
            "distributor_decision_source": decision_source["distributor"],
            "factory_decision_source": decision_source["factory"],

            # Orders/demand received this round.
            "retailer_order_received": order_received["retailer"],
            "wholesaler_order_received": order_received["wholesaler"],
            "distributor_order_received": order_received["distributor"],
            "factory_order_received": order_received["factory"],

            # Physical arrivals this round.
            "retailer_shipment_received": shipment_received["retailer"],
            "wholesaler_shipment_received": shipment_received["wholesaler"],
            "distributor_shipment_received": shipment_received["distributor"],
            "factory_production_received": shipment_received["factory"],

            # Actual physical shipments made this round.
            "retailer_shipped_to_customer": shipped["retailer"],
            "wholesaler_shipped_to_retailer": shipped["wholesaler"],
            "distributor_shipped_to_wholesaler": shipped["distributor"],
            "factory_shipped_to_distributor": shipped["factory"],

            # Generic shipped aliases.
            "retailer_shipped": shipped["retailer"],
            "wholesaler_shipped": shipped["wholesaler"],
            "distributor_shipped": shipped["distributor"],
            "factory_shipped": shipped["factory"],
        }

        for role in roles:
            total_cost, inventory_cost, backlog_cost = costs[role]

            row[f"{role}_inventory"] = agents[role]["inventory"]
            row[f"{role}_backlog"] = agents[role]["backlog"]
            row[f"{role}_effective_inventory"] = (
                agents[role]["inventory"] - agents[role]["backlog"]
            )

            row[f"{role}_cost"] = total_cost
            row[f"{role}_inventory_cost"] = inventory_cost
            row[f"{role}_backlog_cost"] = backlog_cost

            # State used by decision rule before current order was added.
            row[f"{role}_SL_before_order"] = SL_before_order[role]
            row[f"{role}_visible_pipeline_before_order"] = (
                visible_pipeline_before_order[role]
            )

            for eta_label, qty in visible_eta_before_order[role].items():
                row[f"{role}_visible_pipeline_{eta_label}_before_order"] = qty

            # State after new orders and shipments were scheduled.
            row[f"{role}_SL_inbound"] = agents[role]["outstanding_to_supplier"]
            row[f"{role}_order_pipeline"] = pipeline_sum(
                agents[role]["order_pipeline"]
            )
            row[f"{role}_shipment_pipeline"] = pipeline_sum(
                agents[role]["shipment_pipeline"]
            )

            # Backward-compatible alias for earlier analysis code.
            # This is now the inbound physical shipment/production pipeline.
            row[f"{role}_supply_line"] = pipeline_sum(
                agents[role]["shipment_pipeline"]
            )

            row[f"{role}_expected_loss"] = agents[role]["expected_loss"]

        # A more explicit factory alias.
        row["factory_production_pipeline"] = pipeline_sum(
            agents["factory"]["shipment_pipeline"]
        )

        weekly_data.append(row)

    df = pd.DataFrame(weekly_data)

    # ---- Sanity check: warm-up should preserve equilibrium ----

    warmup_sanity_passed = True

    if run_sanity_check and forced_warmup_rounds > 0:
        warmup_df = df[df["week"] < forced_warmup_rounds]

        for role in roles:
            if not (warmup_df[f"{role}_order"] == initial_throughput).all():
                warmup_sanity_passed = False

            if not (warmup_df[f"{role}_inventory"] == initial_inventory).all():
                warmup_sanity_passed = False

            if not (warmup_df[f"{role}_backlog"] == 0).all():
                warmup_sanity_passed = False

            if not (warmup_df[f"{role}_decision_source"] == "forced_warmup").all():
                warmup_sanity_passed = False

        if not warmup_sanity_passed:
            raise AssertionError(
                "Warm-up equilibrium sanity check failed. "
                "Expected forced orders of 4, inventory 12, and backlog 0 "
                "for all roles during warm-up."
            )

    # ---- Aggregates ----

    total_costs = {
        role: float(agents[role]["total_cost"])
        for role in roles
    }

    total_team_cost = float(sum(total_costs.values()))

    # Kept for compatibility with your previous ABM code.
    # Note: after week 10 customer demand is constant, so this becomes factory variance
    # if customer-demand variance is zero.
    if weeks > 20:
        steady = df[df["week"] >= 10]
        cust_var = float(steady["customer_demand"].var())
        fact_var = float(steady["factory_order"].var())
        amplification = (fact_var / cust_var) if cust_var > 0 else fact_var
    else:
        amplification = 0.0

    # Additional full-horizon diagnostic, useful because post-week-10 demand is constant.
    full_cust_var = float(df["customer_demand"].var())
    full_factory_var = float(df["factory_order"].var())
    full_horizon_amplification = (
        full_factory_var / full_cust_var
        if full_cust_var > 0
        else full_factory_var
    )

    orders_placed = {
        role: list(agents[role]["orders_placed"])
        for role in roles
    }

    summary = {
        # Kept for compatibility with previous output.
        # Recalibrate denominators if reporting normalized costs for the 35-round horizon.
        "cost_vs_optimal": total_team_cost / 204.0,
        "cost_vs_human_avg": total_team_cost / 2028.0,

        "max_retailer_order": float(df["retailer_order"].max()),
        "max_wholesaler_order": float(df["wholesaler_order"].max()),
        "max_distributor_order": float(df["distributor_order"].max()),
        "max_factory_order": float(df["factory_order"].max()),

        "min_retailer_inventory": float(df["retailer_inventory"].min()),
        "max_retailer_backlog": float(df["retailer_backlog"].max()),

        "total_backlog_weeks": int(
            (
                df[
                    [
                        "retailer_backlog",
                        "wholesaler_backlog",
                        "distributor_backlog",
                        "factory_backlog",
                    ]
                ].sum(axis=1)
                > 0
            ).sum()
        ),

        "warmup_sanity_passed": warmup_sanity_passed,
    }

    return {
        "data": df,
        "total_costs": total_costs,
        "total_team_cost": total_team_cost,
        "amplification": amplification,
        "full_horizon_amplification": full_horizon_amplification,
        "parameters": {
            "role_parameters": params,
            "sigma_eps": sigma_eps,
            "seed": seed,

            "weeks": weeks,
            "initial_inventory": initial_inventory,

            # New name
            "initial_throughput": initial_throughput,

            # Backward-compatible old name
            "initial_order": initial_throughput,

            "order_delay": order_delay,
            "shipment_lead_time": shipment_lead_time,
            "production_lead_time": production_lead_time,
            "forced_warmup_rounds": forced_warmup_rounds,
            "initial_pipelines_seeded": True,
            "mass_conserving_shipments": True,
            "separate_backlog": True,
            "nonnegative_inventory": True,
        },
        "orders_placed": orders_placed,
        "summary": summary,
    }

## 5. Detailed Simulation Plot (diagnostic)


In [ ]:
def plot_simulation_results(result):
    """Plot orders, effective inventory, and weekly costs with per-role params in the title."""
    data = result["data"]
    weeks = data["week"]

    colors = {
        "customer": "black",
        "retailer": "#1f77b4",
        "wholesaler": "#ff7f0e",
        "distributor": "#2ca02c",
        "factory": "#d62728",
    }

    def _fmt(p):  # compact role param formatter
        return f"θ={p['theta']:.2f}, αS={p['alpha_s']:.2f}, β={p['beta']:.2f}, S′={int(p['s_prime'])}"

    p = result["parameters"]["role_parameters"]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

    # Orders
    ax1.plot(weeks, data["customer_demand"], color=colors["customer"], linewidth=3, linestyle="--", label="Customer Demand", alpha=0.9)
    ax1.plot(weeks, data["retailer_order"], color=colors["retailer"], linewidth=2, label="Retailer", alpha=0.85)
    ax1.plot(weeks, data["wholesaler_order"], color=colors["wholesaler"], linewidth=2, label="Wholesaler", alpha=0.85)
    ax1.plot(weeks, data["distributor_order"], color=colors["distributor"], linewidth=2, label="Distributor", alpha=0.85)
    ax1.plot(weeks, data["factory_order"], color=colors["factory"], linewidth=2, label="Factory", alpha=0.85)
    ax1.set_title("Orders by Supply Chain Level", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Week"); ax1.set_ylabel("Orders (Cases/Week)")
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, max(data["factory_order"].max() * 1.1, 20))

    # Effective Inventory
    re = data["retailer_inventory"]   - data["retailer_backlog"]
    we = data["wholesaler_inventory"] - data["wholesaler_backlog"]
    de = data["distributor_inventory"]- data["distributor_backlog"]
    fe = data["factory_inventory"]    - data["factory_backlog"]

    ax2.plot(weeks, re, color=colors["retailer"], linewidth=2, label="Retailer", alpha=0.85)
    ax2.plot(weeks, we, color=colors["wholesaler"], linewidth=2, label="Wholesaler", alpha=0.85)
    ax2.plot(weeks, de, color=colors["distributor"], linewidth=2, label="Distributor", alpha=0.85)
    ax2.plot(weeks, fe, color=colors["factory"], linewidth=2, label="Factory", alpha=0.85)
    ax2.axhline(y=0,  color="red",  linestyle=":", alpha=0.7, label="Stockout Line")
    ax2.axhline(y=12, color="gray", linestyle=":", alpha=0.5, label="Initial Level")
    ax2.set_title("Effective Inventory (Inventory − Backlog)", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Week"); ax2.set_ylabel("Cases")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    # Weekly Costs (total)
    ax3.plot(weeks, data["retailer_cost"],   color=colors["retailer"],   linewidth=2, label="Retailer",   alpha=0.85)
    ax3.plot(weeks, data["wholesaler_cost"], color=colors["wholesaler"], linewidth=2, label="Wholesaler", alpha=0.85)
    ax3.plot(weeks, data["distributor_cost"],color=colors["distributor"],linewidth=2, label="Distributor",alpha=0.85)
    ax3.plot(weeks, data["factory_cost"],    color=colors["factory"],    linewidth=2, label="Factory",    alpha=0.85)
    ax3.set_title("Weekly Costs", fontsize=14, fontweight="bold")
    ax3.set_xlabel("Week"); ax3.set_ylabel("$/Week")
    ax3.legend(); ax3.grid(True, alpha=0.3)

    # Summary suptitle (restored)
    total_cost = result["total_team_cost"]
    amplification = result["amplification"]
    cost_ratio = result["summary"]["cost_vs_optimal"]

    role_str = " | ".join([
        f"R({_fmt(p['retailer'])})",
        f"W({_fmt(p['wholesaler'])})",
        f"D({_fmt(p['distributor'])})",
        f"F({_fmt(p['factory'])})",
    ])

    fig.suptitle(
        f"Beer Game • Total Cost ${total_cost:.0f} ({cost_ratio:.1f}× optimal) • Amplification {amplification:.1f}×\n{role_str}",
        fontsize=12, fontweight="bold"
    )

    plt.tight_layout()
    plt.show()


## 6. BayesFlow Prior + Simulator Wrapper

Lognormal priors anchored at Sterman's reported means (θ ≈ 0.36, αS ≈ 0.26,
β ≈ 0.34, S′ ≈ 17), with θ and β soft-clipped at 1.0.


In [ ]:
# RNG used by the prior (separate from simulator's RNG)
RNG = np.random.default_rng(2025)

def prior():
    # θ (theta): expectation-formation rate
    theta_r = min(RNG.lognormal(np.log(0.36), 0.6), 1.0)
    theta_d = min(RNG.lognormal(np.log(0.36), 0.6), 1.0)
    theta_w = min(RNG.lognormal(np.log(0.36), 0.6), 1.0)
    theta_f = min(RNG.lognormal(np.log(0.36), 0.6), 1.0)

    # αS (alpha_s): stock-adjustment rate
    alpha_s_r = RNG.lognormal(np.log(0.26), 0.6)
    alpha_s_d = RNG.lognormal(np.log(0.26), 0.6)
    alpha_s_w = RNG.lognormal(np.log(0.26), 0.6)
    alpha_s_f = RNG.lognormal(np.log(0.26), 0.6)

    # β (beta): supply-line recognition (clipped at 1.0)
    beta_r = min(RNG.lognormal(np.log(0.34), 0.6), 1.0)
    beta_d = min(RNG.lognormal(np.log(0.34), 0.6), 1.0)
    beta_w = min(RNG.lognormal(np.log(0.34), 0.6), 1.0)
    beta_f = min(RNG.lognormal(np.log(0.34), 0.6), 1.0)

    # S′ (s_prime): desired stock level
    s_prime_r = RNG.lognormal(np.log(17), 0.6)
    s_prime_d = RNG.lognormal(np.log(17), 0.6)
    s_prime_w = RNG.lognormal(np.log(17), 0.6)
    s_prime_f = RNG.lognormal(np.log(17), 0.6)

    return dict(
        theta_r=theta_r, alpha_s_r=alpha_s_r, beta_r=beta_r, s_prime_r=s_prime_r,
        theta_d=theta_d, alpha_s_d=alpha_s_d, beta_d=beta_d, s_prime_d=s_prime_d,
        theta_w=theta_w, alpha_s_w=alpha_s_w, beta_w=beta_w, s_prime_w=s_prime_w,
        theta_f=theta_f, alpha_s_f=alpha_s_f, beta_f=beta_f, s_prime_f=s_prime_f,
    )

In [ ]:
def simulator_wrapper(theta_r,theta_d, theta_w, theta_f, alpha_s_r, alpha_s_d,alpha_s_w,alpha_s_f, beta_r,beta_d,beta_w,beta_f,s_prime_r,s_prime_d,s_prime_w,s_prime_f):
    simulation_results = simulate_beer_game(
        theta_r=theta_r, alpha_s_r=alpha_s_r, beta_r=beta_r, s_prime_r=s_prime_r,
        theta_d=theta_d, alpha_s_d=alpha_s_d, beta_d=beta_d, s_prime_d=s_prime_d,
        theta_w=theta_w, alpha_s_w=alpha_s_w, beta_w=beta_w, s_prime_w=s_prime_w,
        theta_f=theta_f, alpha_s_f=alpha_s_f, beta_f=beta_f, s_prime_f=s_prime_f,
        weeks=35
    )
    retailer_order = simulation_results['data']['retailer_order']
    wholesaler_order = simulation_results['data']['wholesaler_order']
    distributor_order = simulation_results['data']['distributor_order']
    factory_order = simulation_results['data']['factory_order']
    retailer_inventory = simulation_results['data']['retailer_inventory']
    wholesaler_inventory = simulation_results['data']['wholesaler_inventory']
    distributor_inventory = simulation_results['data']['distributor_inventory']
    factory_inventory = simulation_results['data']['factory_inventory']
    retailer_backlog = simulation_results['data']['retailer_backlog']
    wholesaler_backlog = simulation_results['data']['wholesaler_backlog']
    distributor_backlog = simulation_results['data']['distributor_backlog']
    factory_backlog = simulation_results['data']['factory_backlog']



    simulated_data = {
        'retailer_order' : retailer_order,
        'wholesaler_order' : wholesaler_order,
        'distributor_order' : distributor_order,
        'factory_order' : factory_order,
        'retailer_inventory' : retailer_inventory,
        'wholesaler_inventory' : wholesaler_inventory,
        'distributor_inventory' : distributor_inventory,
        'factory_inventory' : factory_inventory,
        'retailer_backlog' : retailer_backlog,
        'wholesaler_backlog' : wholesaler_backlog,
        'distributor_backlog' : distributor_backlog,
        'factory_backlog' : factory_backlog,
    }


    return simulated_data

## 7. BayesFlow Adapter (parameter constraints + I/O shapes)

The adapter:
- Constrains θ and β to [0, 1]; constrains αS and S′ to [0, ∞).
- Casts everything to float32.
- Marks the 12 observed channels as time series.
- Concatenates parameters into `inference_variables` and channels into `summary_variables`.


In [ ]:
simulator = bf.make_simulator([prior, simulator_wrapper])

adapter = (
    bf.adapters.Adapter()
    .constrain(['theta_r','theta_d','theta_w','theta_f'], lower=0, upper=1,
               inclusive='both', epsilon=1e-9)
    .constrain(['beta_r','beta_d','beta_w','beta_f'], lower=0, upper=1,
               inclusive='both', epsilon=1e-9)
    .constrain(['alpha_s_r','s_prime_r','alpha_s_d','alpha_s_w','alpha_s_f',
                's_prime_d','s_prime_w','s_prime_f'], lower=0, inclusive='lower')
    .convert_dtype("float64", "float32")
    .as_time_series("retailer_order")
    .as_time_series("wholesaler_order")
    .as_time_series("distributor_order")
    .as_time_series("factory_order")
    .as_time_series("retailer_inventory")
    .as_time_series("wholesaler_inventory")
    .as_time_series("distributor_inventory")
    .as_time_series("factory_inventory")
    .as_time_series("retailer_backlog")
    .as_time_series("wholesaler_backlog")
    .as_time_series("distributor_backlog")
    .as_time_series("factory_backlog")
    .concatenate(
        ['theta_r','alpha_s_r','beta_r','s_prime_r',
         'theta_d','alpha_s_d','beta_d','s_prime_d',
         'theta_w','alpha_s_w','beta_w','s_prime_w',
         'theta_f','alpha_s_f','beta_f','s_prime_f'],
        into="inference_variables")
    .concatenate(["retailer_order", 'wholesaler_order', 'distributor_order','factory_order','retailer_inventory','wholesaler_inventory','distributor_inventory','factory_inventory',
                  'retailer_backlog','wholesaler_backlog','distributor_backlog','factory_backlog',
                  ], into="summary_variables")
)

# Smoke test: sample 100 sims and check shapes
adapted_sims = adapter(simulator.sample(100))
print("summary_variables shape:  ", adapted_sims["summary_variables"].shape)
print("inference_variables shape:", adapted_sims["inference_variables"].shape)


## 8. Networks and Workflow


In [ ]:
summary_net   = bf.networks.FusionTransformer(summary_dim=64)
inference_net = bf.networks.CouplingFlow()

workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=inference_net,
    summary_network=summary_net,
)


## 9. Load Trained Approximator


In [ ]:
filepath = MODEL_PATH
approximator = keras.saving.load_model(filepath)


## 10. Calibration Diagnostics (recovery, z-score, ECDF)


In [ ]:
# Posterior draw configuration used everywhere below
num_datasets = 100
num_samples  = 1000

test_sims = workflow.simulate(num_datasets)
samples   = approximator.sample(conditions=test_sims, num_samples=num_samples)

# PLOT_DIR is configured in Section 1 as /results.
PLOT_DIR.mkdir(parents=True, exist_ok=True)

#bf.diagnostics.plots.recovery(samples, test_sims).savefig(PLOT_DIR / "recovery.pdf", format="pdf")
#bf.diagnostics.plots.z_score_contraction(samples, test_sims).savefig(PLOT_DIR / "z_score_contraction.pdf", format="pdf")
#bf.diagnostics.plots.calibration_ecdf(samples, test_sims, difference=True).savefig(PLOT_DIR / "calibration_ecdf.pdf", format="pdf")


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

# This order exactly matches the adapter's 16-dimensional inference vector.
PARAMETER_ORDER = [
    "theta_r", "alpha_s_r", "beta_r", "s_prime_r",
    "theta_d", "alpha_s_d", "beta_d", "s_prime_d",
    "theta_w", "alpha_s_w", "beta_w", "s_prime_w",
    "theta_f", "alpha_s_f", "beta_f", "s_prime_f",
]

ROLE_ORDER_PPC = [
    "retailer",
    "wholesaler",
    "distributor",
    "factory",
]

# The 12 channels used to train the INN.
OBSERVABLE_KEYS_PPC = [
    f"{role}_{variable}"
    for variable in ("order", "inventory", "backlog")
    for role in ROLE_ORDER_PPC
]


def posterior_samples_to_frame(samples):
    """
    Convert one BayesFlow posterior sample object to a clean DataFrame
    with the 16 parameter columns in adapter order.
    """
    df = workflow.samples_to_data_frame(samples).copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join(str(x) for x in col if str(x))
            for col in df.columns
        ]

    if set(PARAMETER_ORDER).issubset(df.columns):
        out = df[PARAMETER_ORDER].copy()
    elif df.shape[1] == len(PARAMETER_ORDER):
        # Fallback for unnamed or numerically named BayesFlow columns.
        # The adapter concatenates the variables in PARAMETER_ORDER.
        out = df.copy()
        out.columns = PARAMETER_ORDER
    else:
        raise ValueError(
            "Could not identify the 16 posterior parameter columns. "
            f"Found columns: {list(df.columns)}"
        )

    return (
        out
        .apply(pd.to_numeric, errors="coerce")
        .dropna(how="any")
        .reset_index(drop=True)
    )


def infer_posterior_frame(
    observed,
    *,
    num_samples=2000,
    seed=42,
):
    """
    Infer one joint posterior distribution for one complete
    four-role Beer Game session.
    """
    try:
        raw = approximator.sample(
            conditions=observed,
            num_samples=num_samples,
            seed=seed,
        )
    except TypeError:
        # Compatibility fallback if the loaded approximator does not
        # expose a seed argument.
        keras.utils.set_random_seed(seed)
        raw = approximator.sample(
            conditions=observed,
            num_samples=num_samples,
        )

    return posterior_samples_to_frame(raw)


def effective_parameter_dict(params):
    """
    Apply the same parameter support used by the simulator and adapter.
    """
    out = {
        name: float(params[name])
        for name in PARAMETER_ORDER
    }

    for name in PARAMETER_ORDER:
        if name.startswith("theta_") or name.startswith("beta_"):
            out[name] = float(np.clip(out[name], 0.0, 1.0))
        else:
            out[name] = float(max(0.0, out[name]))

    return out


def simulate_observables_from_params(
    params,
    *,
    seed=42,
    sigma_eps=1.0,
):
    """
    Run the mass-conserving Beer Game simulator and return exactly the
    12 observable channels supplied to the INN.
    """
    params = effective_parameter_dict(params)

    result = simulate_beer_game(
        **params,
        sigma_eps=sigma_eps,
        seed=int(seed),
        weeks=35,
    )

    data = result["data"]

    return {
        key: np.asarray(data[key], dtype=float)
        for key in OBSERVABLE_KEYS_PPC
    }


def draw_posterior_predictive(
    posterior_df,
    *,
    n_draws=250,
    seed=42,
    sigma_eps=1.0,
    simulator_seed=42,
    vary_simulator_seed=False,
):
    """
    Generate posterior predictive trajectories.

    The currently trained BayesFlow simulator uses the default
    simulate_beer_game(seed=42), because simulator_wrapper does not pass
    a seed. Consequently, the default here also fixes simulator_seed=42.
    The predictive bands then represent posterior parameter uncertainty.

    Set vary_simulator_seed=True only after retraining the INN with
    randomized simulator seeds.
    """
    if posterior_df.empty:
        raise ValueError("posterior_df is empty")

    rng = np.random.default_rng(seed)

    draw_indices = rng.choice(
        len(posterior_df),
        size=int(n_draws),
        replace=(n_draws > len(posterior_df)),
    )

    predictive = {
        key: []
        for key in OBSERVABLE_KEYS_PPC
    }

    for idx in draw_indices:
        params = (
            posterior_df
            .iloc[int(idx)][PARAMETER_ORDER]
            .to_dict()
        )

        seed_for_simulation = (
            int(rng.integers(0, 2**31 - 1))
            if vary_simulator_seed
            else int(simulator_seed)
        )

        simulated = simulate_observables_from_params(
            params,
            seed=seed_for_simulation,
            sigma_eps=sigma_eps,
        )

        for key in OBSERVABLE_KEYS_PPC:
            predictive[key].append(simulated[key])

    return {
        key: np.stack(values, axis=0)
        for key, values in predictive.items()
    }


def summarize_posterior_predictive(
    observed,
    predictive,
    *,
    label,
    interval=0.90,
):
    """
    Calculate pointwise predictive coverage and trajectory errors for
    orders, inventories, and backlogs in all four roles.
    """
    alpha = 1.0 - float(interval)
    rows = []

    for key in OBSERVABLE_KEYS_PPC:
        observed_values = np.asarray(
            observed[key],
            dtype=float,
        )
        predicted_values = np.asarray(
            predictive[key],
            dtype=float,
        )

        if (
            predicted_values.ndim != 2
            or predicted_values.shape[1] != observed_values.shape[0]
        ):
            raise ValueError(
                f"Shape mismatch for {key}: "
                f"observed={observed_values.shape}, "
                f"predictive={predicted_values.shape}"
            )

        lower = np.quantile(
            predicted_values,
            alpha / 2.0,
            axis=0,
        )
        median = np.quantile(
            predicted_values,
            0.50,
            axis=0,
        )
        upper = np.quantile(
            predicted_values,
            1.0 - alpha / 2.0,
            axis=0,
        )

        role, variable = key.rsplit("_", 1)

        rows.append({
            "label": label,
            "role": role,
            "variable": variable,
            "interval": interval,
            "pointwise_coverage": float(
                np.mean(
                    (observed_values >= lower)
                    & (observed_values <= upper)
                )
            ),
            "rmse_to_predictive_median": float(
                np.sqrt(
                    np.mean(
                        (observed_values - median) ** 2
                    )
                )
            ),
            "mae_to_predictive_median": float(
                np.mean(
                    np.abs(observed_values - median)
                )
            ),
            "mean_interval_width": float(
                np.mean(upper - lower)
            ),
        })

    return pd.DataFrame(rows)


def make_order_ppc_figure(
    observed,
    predictive,
    *,
    title,
    outer_interval=0.90,
    inner_interval=0.50,
):
    """
    Plot observed orders against the posterior predictive distribution
    separately for Retailer, Wholesaler, Distributor, and Factory.
    """
    outer_alpha = 1.0 - outer_interval
    inner_alpha = 1.0 - inner_interval

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(13, 8),
        sharex=True,
    )
    axes = axes.ravel()

    for ax, role in zip(axes, ROLE_ORDER_PPC):
        key = f"{role}_order"

        observed_values = np.asarray(
            observed[key],
            dtype=float,
        )
        predicted_values = np.asarray(
            predictive[key],
            dtype=float,
        )

        rounds = np.arange(
            1,
            len(observed_values) + 1,
        )

        outer_lower = np.quantile(
            predicted_values,
            outer_alpha / 2.0,
            axis=0,
        )
        outer_upper = np.quantile(
            predicted_values,
            1.0 - outer_alpha / 2.0,
            axis=0,
        )

        inner_lower = np.quantile(
            predicted_values,
            inner_alpha / 2.0,
            axis=0,
        )
        inner_upper = np.quantile(
            predicted_values,
            1.0 - inner_alpha / 2.0,
            axis=0,
        )

        predictive_median = np.quantile(
            predicted_values,
            0.50,
            axis=0,
        )

        ax.fill_between(
            rounds,
            outer_lower,
            outer_upper,
            alpha=0.18,
            label=f"{int(outer_interval * 100)}% predictive interval",
        )
        ax.fill_between(
            rounds,
            inner_lower,
            inner_upper,
            alpha=0.30,
            label=f"{int(inner_interval * 100)}% predictive interval",
        )
        ax.plot(
            rounds,
            predictive_median,
            linewidth=2.0,
            label="Posterior predictive median",
        )
        ax.plot(
            rounds,
            observed_values,
            linewidth=1.8,
            linestyle="--",
            label="Observed / target trajectory",
        )

        ax.set_title(role.capitalize())
        ax.set_xlabel("Round")
        ax.set_ylabel("Order")
        ax.grid(alpha=0.25)

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=2,
        frameon=False,
    )
    fig.suptitle(title, fontsize=14)
    fig.tight_layout(
        rect=(0, 0.08, 1, 0.95)
    )

    return fig

## 11. Data Loaders

Two ways to produce a 12-channel observation dict:
- `load_data(...)` runs the simulator with explicit role params.
- `load_simulated_data(...)` reads a `*_extracted.json` from an LLM run.


In [ ]:
def load_data(theta_r,theta_d, theta_w, theta_f, alpha_s_r, alpha_s_d,alpha_s_w,alpha_s_f, beta_r,beta_d,beta_w,beta_f,s_prime_r,s_prime_d,s_prime_w,s_prime_f):
    simulation_results = simulate_beer_game(
        theta_r=theta_r, alpha_s_r=alpha_s_r, beta_r=beta_r, s_prime_r=s_prime_r,
        theta_d=theta_d, alpha_s_d=alpha_s_d, beta_d=beta_d, s_prime_d=s_prime_d,
        theta_w=theta_w, alpha_s_w=alpha_s_w, beta_w=beta_w, s_prime_w=s_prime_w,
        theta_f=theta_f, alpha_s_f=alpha_s_f, beta_f=beta_f, s_prime_f=s_prime_f,
        weeks=35
    )
    retailer_order = simulation_results['data']['retailer_order']
    wholesaler_order = simulation_results['data']['wholesaler_order']
    distributor_order = simulation_results['data']['distributor_order']
    factory_order = simulation_results['data']['factory_order']
    retailer_inventory = simulation_results['data']['retailer_inventory']
    wholesaler_inventory = simulation_results['data']['wholesaler_inventory']
    distributor_inventory = simulation_results['data']['distributor_inventory']
    factory_inventory = simulation_results['data']['factory_inventory']
    retailer_backlog = simulation_results['data']['retailer_backlog']
    wholesaler_backlog = simulation_results['data']['wholesaler_backlog']
    distributor_backlog = simulation_results['data']['distributor_backlog']
    factory_backlog = simulation_results['data']['factory_backlog']



    simulated_data = {
        'retailer_order' : retailer_order,
        'wholesaler_order' : wholesaler_order,
        'distributor_order' : distributor_order,
        'factory_order' : factory_order,
        'retailer_inventory' : retailer_inventory,
        'wholesaler_inventory' : wholesaler_inventory,
        'distributor_inventory' : distributor_inventory,
        'factory_inventory' : factory_inventory,
        'retailer_backlog' : retailer_backlog,
        'wholesaler_backlog' : wholesaler_backlog,
        'distributor_backlog' : distributor_backlog,
        'factory_backlog' : factory_backlog,
    }


    return simulated_data


REQUIRED_KEYS: List[str] = [
    "retailer_order", "wholesaler_order", "distributor_order", "factory_order",
    "retailer_inventory", "wholesaler_inventory", "distributor_inventory", "factory_inventory",
    "retailer_backlog", "wholesaler_backlog", "distributor_backlog", "factory_backlog",
]

def load_simulated_data(extracted_dir: str, sim_name: Optional[str] = None) -> Dict[str, Any]:
    """
    Load a *_extracted.json file produced by an LLM beer-game run.
    If `sim_name` is None, picks the first match in alphabetical order.
    Coerces list fields into int NumPy arrays.
    """
    extracted_dir_abs = os.path.abspath(extracted_dir)

    if sim_name:
        file_path = os.path.join(extracted_dir_abs, f"{sim_name}_extracted.json")
        if not os.path.isfile(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")
    else:
        matches = sorted(glob(os.path.join(extracted_dir_abs, "*_extracted.json")))
        if not matches:
            raise FileNotFoundError(f"No '*_extracted.json' files in: {extracted_dir_abs}")
        file_path = matches[0]

    with open(file_path, "r") as f:
        payload: Dict[str, Any] = json.load(f)

    missing = [k for k in REQUIRED_KEYS if k not in payload]
    if missing:
        raise ValueError(f"{os.path.basename(file_path)} missing keys: {', '.join(missing)}")

    out: Dict[str, Any] = {}
    for key in REQUIRED_KEYS:
        value = payload[key]
        if isinstance(value, list):
            try:
                out[key] = np.array(value, dtype=int)
            except Exception:
                out[key] = np.array(value, dtype=object)
        else:
            out[key] = value
    return out


## 12. Quick check: posterior given Sterman default params


In [ ]:
obs_cases = load_data(
    theta_r=0.36, alpha_s_r=0.26, beta_r=0.34, s_prime_r=17,
    theta_w=0.36, alpha_s_w=0.26, beta_w=0.34, s_prime_w=17,
    theta_d=0.36, alpha_s_d=0.26, beta_d=0.34, s_prime_d=17,
    theta_f=0.36, alpha_s_f=0.26, beta_f=0.34, s_prime_f=17,
)
samples_check = approximator.sample(conditions=obs_cases, num_samples=num_samples)
samples_check = workflow.samples_to_data_frame(samples_check)
samples_check.mean()


## 13. Sterman's 11 Human Experiments (reference table)

Encoded as (θ, αS, β, S′) for each of the 4 roles. R² and RMSE are dropped.


In [ ]:
_STERMAN_11 = [
    # 0 Bassbeer
    {"Retailer":   {"theta": 0.90, "alpha_s": 0.10, "beta": 0.65, "s_prime": 20},
     "Wholesaler": {"theta": 0.00, "alpha_s": 0.25, "beta": 0.50, "s_prime": 27},
     "Distributor":{"theta": 0.15, "alpha_s": 0.05, "beta": 0.35, "s_prime": 14},
     "Factory":    {"theta": 1.00, "alpha_s": 0.65, "beta": 0.40, "s_prime": 15}},
    # 1 Budweiser
    {"Retailer":   {"theta": 0.00, "alpha_s": 0.40, "beta": 0.10, "s_prime": 7},
     "Wholesaler": {"theta": 0.00, "alpha_s": 0.40, "beta": 0.75, "s_prime": 30},
     "Distributor":{"theta": 0.00, "alpha_s": 0.30, "beta": 0.10, "s_prime": 10},
     "Factory":    {"theta": 0.25, "alpha_s": 0.25, "beta": 0.10, "s_prime": 9}},
    # 2 Coors
    {"Retailer":   {"theta": 0.00, "alpha_s": 0.20, "beta": 0.00, "s_prime": 25},
     "Wholesaler": {"theta": 0.00, "alpha_s": 0.15, "beta": 0.50, "s_prime": 38},
     "Distributor":{"theta": 0.90, "alpha_s": 0.30, "beta": 0.20, "s_prime": 10},
     "Factory":    {"theta": 0.25, "alpha_s": 0.30, "beta": 0.00, "s_prime": 18}},
    # 3 Freebeer
    {"Retailer":   {"theta": 0.40, "alpha_s": 0.35, "beta": 0.45, "s_prime": 15},
     "Wholesaler": {"theta": 0.30, "alpha_s": 0.05, "beta": 0.00, "s_prime": 30},
     "Distributor":{"theta": 0.05, "alpha_s": 0.35, "beta": 1.00, "s_prime": 18},
     "Factory":    {"theta": 0.25, "alpha_s": 0.25, "beta": 0.00, "s_prime": 19}},
    # 4 Grin & Beer It
    {"Retailer":   {"theta": 0.10, "alpha_s": 0.35, "beta": 0.65, "s_prime": 13},
     "Wholesaler": {"theta": 0.95, "alpha_s": 0.15, "beta": 0.55, "s_prime": 14},
     "Distributor":{"theta": 0.20, "alpha_s": 0.20, "beta": 0.30, "s_prime": 19},
     "Factory":    {"theta": 0.25, "alpha_s": 0.35, "beta": 0.55, "s_prime": 24}},
    # 5 Grizzly
    {"Retailer":   {"theta": 0.05, "alpha_s": 0.30, "beta": 0.65, "s_prime": 31},
     "Wholesaler": {"theta": 0.30, "alpha_s": 0.20, "beta": 0.35, "s_prime": 27},
     "Distributor":{"theta": 0.15, "alpha_s": 0.05, "beta": 0.25, "s_prime": 15},
     "Factory":    {"theta": 0.55, "alpha_s": 0.65, "beta": 0.00, "s_prime": 9}},
    # 6 Heineken1
    {"Retailer":   {"theta": 0.95, "alpha_s": 0.15, "beta": 0.00, "s_prime": 9},
     "Wholesaler": {"theta": 0.50, "alpha_s": 0.00, "beta": 0.34, "s_prime": 17},
     "Distributor":{"theta": 0.20, "alpha_s": 0.30, "beta": 0.05, "s_prime": 8},
     "Factory":    {"theta": 0.80, "alpha_s": 0.00, "beta": 0.34, "s_prime": 17}},
    # 7 Heineken2
    {"Retailer":   {"theta": 0.50, "alpha_s": 0.05, "beta": 0.60, "s_prime": 6},
     "Wholesaler": {"theta": 0.40, "alpha_s": 0.10, "beta": 0.30, "s_prime": 16},
     "Distributor":{"theta": 1.00, "alpha_s": 0.15, "beta": 0.60, "s_prime": 14},
     "Factory":    {"theta": 0.55, "alpha_s": 0.80, "beta": 0.00, "s_prime": 9}},
    # 8 Heineken3
    {"Retailer":   {"theta": 0.05, "alpha_s": 0.30, "beta": 0.45, "s_prime": 5},
     "Wholesaler": {"theta": 0.20, "alpha_s": 0.00, "beta": 0.34, "s_prime": 17},
     "Distributor":{"theta": 0.30, "alpha_s": 0.10, "beta": 0.30, "s_prime": 12},
     "Factory":    {"theta": 0.00, "alpha_s": 0.30, "beta": 0.15, "s_prime": 17}},
    # 9 Suds
    {"Retailer":   {"theta": 1.00, "alpha_s": 0.00, "beta": 0.34, "s_prime": 17},
     "Wholesaler": {"theta": 0.05, "alpha_s": 0.30, "beta": 0.20, "s_prime": 20},
     "Distributor":{"theta": 0.15, "alpha_s": 0.60, "beta": 0.35, "s_prime": 0},
     "Factory":    {"theta": 0.40, "alpha_s": 0.35, "beta": 1.05, "s_prime": 32}},
    # 10 Twoborg
    {"Retailer":   {"theta": 0.75, "alpha_s": 0.35, "beta": 0.00, "s_prime": 4},
     "Wholesaler": {"theta": 0.00, "alpha_s": 0.25, "beta": 0.05, "s_prime": 18},
     "Distributor":{"theta": 0.05, "alpha_s": 0.50, "beta": 0.00, "s_prime": 15},
     "Factory":    {"theta": 0.95, "alpha_s": 0.30, "beta": 0.20, "s_prime": 26}},
]

_ROLE_SUFFIX = {"Retailer":"r", "Distributor":"d", "Wholesaler":"w", "Factory":"f"}


def build_human_experiments_df():
    """Convert _STERMAN_11 into a (params x sim_*) DataFrame matching model dirs."""
    rows = [
        "theta_r","alpha_s_r","beta_r","s_prime_r",
        "theta_d","alpha_s_d","beta_d","s_prime_d",
        "theta_w","alpha_s_w","beta_w","s_prime_w",
        "theta_f","alpha_s_f","beta_f","s_prime_f",
    ]
    cols = [f"sim_{i}" for i in range(11)]
    df = pd.DataFrame(index=rows, columns=cols, dtype=float)

    for i, exp in enumerate(_STERMAN_11):
        for role, params in exp.items():
            sfx = _ROLE_SUFFIX[role]
            df.loc[f"theta_{sfx}",    f"sim_{i}"] = params["theta"]
            df.loc[f"alpha_s_{sfx}",  f"sim_{i}"] = params["alpha_s"]
            df.loc[f"beta_{sfx}",     f"sim_{i}"] = params["beta"]
            df.loc[f"s_prime_{sfx}",  f"sim_{i}"] = params["s_prime"]
    return df


human_df_sims     = build_human_experiments_df()
human_mean_series = human_df_sims.mean(axis=1)


In [ ]:
# Keep the reported/imputed table separately. For inference-space
# comparisons, use beta=1.00 for Suds-Factory because both the simulator
# and BayesFlow adapter constrain beta to [0, 1].
human_df_sims_reported_or_imputed = build_human_experiments_df()

human_df_sims = (
    human_df_sims_reported_or_imputed
    .copy()
)
human_df_sims.loc["beta_f", "sim_9"] = 1.00

human_mean_series = human_df_sims.mean(axis=1)


STERMAN_SESSION_NAMES = [
    "Bassbeer",
    "Budweiser",
    "Coors",
    "Freebeer",
    "Grin_and_Beer_It",
    "Grizzly",
    "Heineken1",
    "Heineken2",
    "Heineken3",
    "Suds",
    "Twoborg",
]

STERMAN_ROLE_TO_SUFFIX = {
    "Retailer": "r",
    "Wholesaler": "w",
    "Distributor": "d",
    "Factory": "f",
}

STERMAN_ROLE_LABELS = {
    "r": "Retailer",
    "w": "Wholesaler",
    "d": "Distributor",
    "f": "Factory",
}

STERMAN_ROLE_MARKERS = {
    "r": "o",
    "w": "s",
    "d": "^",
    "f": "D",
}

STERMAN_PARAMETER_LABELS = {
    "theta": r"$\theta$",
    "alpha_s": r"$\alpha_S$",
    "beta": r"$\beta$",
    "s_prime": r"$S'$",
}


def sterman_session_to_parameter_dict(session):
    """
    Flatten one nested Sterman session into the 16 names used by the
    BayesFlow adapter.
    """
    flattened = {}

    for role_name, role_parameters in session.items():
        suffix = STERMAN_ROLE_TO_SUFFIX[role_name]

        flattened[f"theta_{suffix}"] = float(
            role_parameters["theta"]
        )
        flattened[f"alpha_s_{suffix}"] = float(
            role_parameters["alpha_s"]
        )
        flattened[f"beta_{suffix}"] = float(
            role_parameters["beta"]
        )
        flattened[f"s_prime_{suffix}"] = float(
            role_parameters["s_prime"]
        )

    return flattened


def parameter_family(parameter_name):
    """Return theta, alpha_s, beta, or s_prime."""
    if parameter_name.startswith("theta_"):
        return "theta"
    if parameter_name.startswith("alpha_s_"):
        return "alpha_s"
    if parameter_name.startswith("beta_"):
        return "beta"
    if parameter_name.startswith("s_prime_"):
        return "s_prime"

    raise ValueError(
        f"Unknown parameter name: {parameter_name}"
    )


def parameter_at_support_boundary(
    parameter_name,
    value,
):
    """
    Identify values at the boundary of the simulator/adapter support.
    Boundary values are useful to report separately because the lognormal
    training prior places its mass in the positive interior.
    """
    value = float(value)

    if parameter_name.startswith("theta_"):
        return bool(
            np.isclose(value, 0.0)
            or np.isclose(value, 1.0)
        )

    if parameter_name.startswith("beta_"):
        return bool(
            np.isclose(value, 0.0)
            or np.isclose(value, 1.0)
        )

    return bool(np.isclose(value, 0.0))


def run_exact_sterman_recovery(
    *,
    posterior_samples=2000,
    generating_seed=42,
    posterior_seed=2026,
):
    """
    Infer the 11 Sterman parameter sets using the already-trained INN.

    Recovery is evaluated against the effective generating values used by
    the simulator. This distinction affects Suds-Factory beta, for which
    the reported value 1.05 is mapped to the supported value 1.00.
    """
    rng = np.random.default_rng(
        posterior_seed
    )

    recovery_rows = []
    posterior_by_session = {}
    observed_by_session = {}

    for session_id, (session_name, session) in enumerate(
        zip(
            STERMAN_SESSION_NAMES,
            _STERMAN_11,
        )
    ):
        reported_or_imputed = (
            sterman_session_to_parameter_dict(
                session
            )
        )
        generating_parameters = (
            effective_parameter_dict(
                reported_or_imputed
            )
        )

        # This reproduces the simulator convention used by load_data()
        # and by the network's training simulator.
        observed = simulate_observables_from_params(
            generating_parameters,
            seed=generating_seed,
            sigma_eps=1.0,
        )

        posterior = infer_posterior_frame(
            observed,
            num_samples=posterior_samples,
            seed=int(
                rng.integers(
                    0,
                    2**31 - 1,
                )
            ),
        )

        observed_by_session[session_name] = observed
        posterior_by_session[session_name] = posterior

        for parameter in PARAMETER_ORDER:
            values = posterior[
                parameter
            ].to_numpy(dtype=float)

            role_suffix = parameter.rsplit(
                "_",
                1,
            )[1]

            lower_90 = float(
                np.quantile(values, 0.05)
            )
            upper_90 = float(
                np.quantile(values, 0.95)
            )

            true_value = generating_parameters[
                parameter
            ]

            recovery_rows.append({
                "session_id": session_id,
                "session": session_name,
                "role_suffix": role_suffix,
                "role": STERMAN_ROLE_LABELS[
                    role_suffix
                ],
                "parameter": parameter,
                "parameter_family": parameter_family(
                    parameter
                ),
                "reported_or_imputed_value": (
                    reported_or_imputed[
                        parameter
                    ]
                ),
                "generating_value": true_value,
                "boundary_adjusted": not np.isclose(
                    reported_or_imputed[
                        parameter
                    ],
                    true_value,
                ),
                "at_support_boundary": (
                    parameter_at_support_boundary(
                        parameter,
                        true_value,
                    )
                ),
                "posterior_mean": float(
                    np.mean(values)
                ),
                "posterior_median": float(
                    np.median(values)
                ),
                "q05": lower_90,
                "q95": upper_90,
                "covered_90": bool(
                    lower_90
                    <= true_value
                    <= upper_90
                ),
            })

    recovery = pd.DataFrame(
        recovery_rows
    )

    return (
        recovery,
        posterior_by_session,
        observed_by_session,
    )


def summarize_sterman_recovery(recovery):
    """
    Aggregate exact-Sterman recovery separately for theta, alpha_s,
    beta, and s_prime.
    """
    summary_rows = []

    for family, subset in recovery.groupby(
        "parameter_family",
        sort=False,
    ):
        true_values = subset[
            "generating_value"
        ].to_numpy(dtype=float)

        estimated_values = subset[
            "posterior_median"
        ].to_numpy(dtype=float)

        if (
            len(true_values) > 1
            and np.std(true_values) > 0
            and np.std(estimated_values) > 0
        ):
            correlation = float(
                np.corrcoef(
                    true_values,
                    estimated_values,
                )[0, 1]
            )
        else:
            correlation = np.nan

        summary_rows.append({
            "parameter_family": family,
            "n_role_session_values": int(
                len(subset)
            ),
            "pearson_r": correlation,
            "mae": float(
                np.mean(
                    np.abs(
                        estimated_values
                        - true_values
                    )
                )
            ),
            "rmse": float(
                np.sqrt(
                    np.mean(
                        (
                            estimated_values
                            - true_values
                        ) ** 2
                    )
                )
            ),
            "coverage_90": float(
                subset[
                    "covered_90"
                ].mean()
            ),
        })

    return pd.DataFrame(
        summary_rows
    )


def plot_exact_sterman_recovery(
    recovery,
    *,
    savepath,
):
    """
    Plot effective generating values against posterior medians for all
    11 sessions and four roles.
    """
    families = [
        "theta",
        "alpha_s",
        "beta",
        "s_prime",
    ]

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(12, 10),
    )

    for ax, family in zip(
        axes.ravel(),
        families,
    ):
        subset = recovery[
            recovery["parameter_family"]
            == family
        ]

        for role_suffix, role_subset in subset.groupby(
            "role_suffix",
            sort=False,
        ):
            x = role_subset[
                "generating_value"
            ].to_numpy(dtype=float)

            y = role_subset[
                "posterior_median"
            ].to_numpy(dtype=float)

            y_error = np.vstack([
                y
                - role_subset[
                    "q05"
                ].to_numpy(dtype=float),
                role_subset[
                    "q95"
                ].to_numpy(dtype=float)
                - y,
            ])

            ax.errorbar(
                x,
                y,
                yerr=y_error,
                fmt=STERMAN_ROLE_MARKERS[
                    role_suffix
                ],
                linestyle="none",
                capsize=2,
                alpha=0.75,
                label=STERMAN_ROLE_LABELS[
                    role_suffix
                ],
            )

        plotted_values = np.concatenate([
            subset[
                "generating_value"
            ].to_numpy(dtype=float),
            subset[
                "q05"
            ].to_numpy(dtype=float),
            subset[
                "q95"
            ].to_numpy(dtype=float),
        ])

        minimum = float(
            np.nanmin(plotted_values)
        )
        maximum = float(
            np.nanmax(plotted_values)
        )
        padding = 0.05 * (
            maximum - minimum
            if maximum > minimum
            else 1.0
        )

        ax.plot(
            [
                minimum - padding,
                maximum + padding,
            ],
            [
                minimum - padding,
                maximum + padding,
            ],
            linestyle="--",
            linewidth=1.2,
        )

        ax.set_xlim(
            minimum - padding,
            maximum + padding,
        )
        ax.set_ylim(
            minimum - padding,
            maximum + padding,
        )
        ax.set_title(
            STERMAN_PARAMETER_LABELS[
                family
            ]
        )
        ax.set_xlabel(
            "Effective generating value"
        )
        ax.set_ylabel(
            "Posterior median (90% interval)"
        )
        ax.grid(alpha=0.25)

    handles, labels = (
        axes
        .ravel()[0]
        .get_legend_handles_labels()
    )

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=4,
        frameon=False,
    )
    fig.suptitle(
        "Recovery of Sterman-reported parameter sets",
        fontsize=14,
    )
    fig.tight_layout(
        rect=(0, 0.06, 1, 0.96)
    )
    fig.savefig(
        savepath,
        bbox_inches="tight",
    )

    return fig


# ---------------------------------------------------------------------
# Run the exact parameter-recovery analysis
# ---------------------------------------------------------------------

STERMAN_POSTERIOR_SAMPLES = 5000
STERMAN_PPC_DRAWS = 1000

(
    sterman_recovery,
    sterman_posteriors,
    sterman_observed,
) = run_exact_sterman_recovery(
    posterior_samples=STERMAN_POSTERIOR_SAMPLES,
    generating_seed=42,
    posterior_seed=2026,
)

# Report all values and, separately, values away from support boundaries.
sterman_recovery_metrics = pd.concat(
    [
        summarize_sterman_recovery(
            sterman_recovery
        ).assign(
            subset="all values"
        ),
        summarize_sterman_recovery(
            sterman_recovery.loc[
                ~sterman_recovery[
                    "at_support_boundary"
                ]
            ]
        ).assign(
            subset="interior values only"
        ),
    ],
    ignore_index=True,
)

sterman_recovery.to_csv(
    PLOT_DIR
    / "sterman_exact_parameter_recovery.csv",
    index=False,
)

sterman_recovery_metrics.to_csv(
    PLOT_DIR
    / "sterman_exact_parameter_recovery_metrics.csv",
    index=False,
)

recovery_figure = (
    plot_exact_sterman_recovery(
        sterman_recovery,
        savepath=(
            PLOT_DIR
            / "sterman_exact_parameter_recovery.pdf"
        ),
    )
)
plt.show()
plt.close(recovery_figure)

display(
    sterman_recovery_metrics
)

# This table should identify Suds-Factory beta:
# reported value 1.05, effective generating value 1.00.
display(
    sterman_recovery.loc[
        sterman_recovery[
            "boundary_adjusted"
        ],
        [
            "session",
            "role",
            "parameter",
            "reported_or_imputed_value",
            "generating_value",
        ],
    ]
)


# ---------------------------------------------------------------------
# Reviewer check 2:
# Posterior predictive checks for all 11 Sterman-generated sessions
# ---------------------------------------------------------------------

sterman_ppc_metrics = []

with PdfPages(
    PLOT_DIR
    / "ppc_sterman_all_sessions.pdf"
) as pdf:

    for session_id, session_name in enumerate(
        STERMAN_SESSION_NAMES
    ):
        predictive = draw_posterior_predictive(
            sterman_posteriors[
                session_name
            ],
            n_draws=STERMAN_PPC_DRAWS,
            seed=40_000 + session_id,
            sigma_eps=1.0,
            simulator_seed=42,
            vary_simulator_seed=False,
        )

        metrics = summarize_posterior_predictive(
            sterman_observed[
                session_name
            ],
            predictive,
            label=session_name,
            interval=0.90,
        )
        metrics.insert(
            0,
            "source",
            "Sterman-generated",
        )
        sterman_ppc_metrics.append(
            metrics
        )

        figure = make_order_ppc_figure(
            sterman_observed[
                session_name
            ],
            predictive,
            title=(
                "Sterman posterior predictive check: "
                f"{session_name}"
            ),
        )

        # Multipage PDF containing all 11 sessions.
        pdf.savefig(
            figure,
            bbox_inches="tight",
        )

        # Individual PDFs, allowing selected sessions to be placed
        # directly in the appendix.
        figure.savefig(
            PLOT_DIR
            / (
                "ppc_sterman_"
                f"{session_name.lower()}.pdf"
            ),
            bbox_inches="tight",
        )

        plt.close(figure)

sterman_ppc_metrics = pd.concat(
    sterman_ppc_metrics,
    ignore_index=True,
)

sterman_ppc_metrics.to_csv(
    PLOT_DIR
    / "ppc_sterman_metrics.csv",
    index=False,
)

# Order-trajectory PPC summary across the 11 Sterman sessions.
display(
    sterman_ppc_metrics.loc[
        sterman_ppc_metrics[
            "variable"
        ] == "order"
    ]
    .groupby("role")[
        [
            "pointwise_coverage",
            "rmse_to_predictive_median",
            "mae_to_predictive_median",
            "mean_interval_width",
        ]
    ]
    .mean()
    .round(3)
)

## 14. Append Human Reference to BayesFlow Outputs


In [ ]:
def append_humans_to_outputs(all_samples_by_dir, overall_means_by_dir):
    """
    Tag the 11 Sterman experiments as a synthetic directory named "human"
    so they appear alongside model outputs in plots and tables.
    """
    human_stacked = human_df_sims.copy()
    human_stacked["directory"] = "human"
    human_stacked["parameter"] = human_stacked.index
    human_stacked = human_stacked.set_index(["directory", "parameter"])

    all_with_human   = pd.concat([all_samples_by_dir, human_stacked], axis=0)
    means_with_human = overall_means_by_dir.copy()
    means_with_human["human"] = human_mean_series
    return all_with_human, means_with_human


## 15. Compute Posterior Means per Directory (with outlier handling)

For each directory in `dirs`:
1. Load each `sim_{i}_extracted.json`.
2. Sample `num_samples` posterior draws.
3. Optionally drop / winsorize per-parameter outliers across draws.
4. Aggregate to a per-sim mean (or median if too few draws survive).


In [ ]:
def compute_means_for_dirs(
    dirs,
    sim_indices=range(10),
    *,
    verbose=True,
    # ---- per-parameter outlier controls applied to posterior draws ----
    outlier_mode='iqr',   # 'none' | 'iqr' | 'zscore' | 'mad' | 'winsor'
    iqr_k=1,
    z_thresh=3.0,
    mad_k=3.5,
    min_valid=3,
    use_median_if_sparse=False,
):
    def _filter_or_winsorize(col):
        x = col.astype(float).replace([np.inf, -np.inf], np.nan)
        if outlier_mode == 'none':
            return x.dropna()

        v = x.values
        mask = np.isfinite(v)
        v_clean = v[mask]
        if v_clean.size == 0:
            return x.dropna()

        if outlier_mode in ('iqr', 'winsor'):
            q1, q3 = np.percentile(v_clean, [25, 75])
            iqr = q3 - q1
            lo, hi = q1 - iqr_k * iqr, q3 + iqr_k * iqr
            if outlier_mode == 'winsor':
                v_new = v.copy()
                v_new[mask] = np.clip(v_clean, lo, hi)
                return pd.Series(v_new, index=x.index).dropna()
            return x[(x >= lo) & (x <= hi)].dropna()

        if outlier_mode == 'zscore':
            mu, sd = np.nanmean(v_clean), np.nanstd(v_clean, ddof=1)
            if sd == 0 or not np.isfinite(sd):
                return x.dropna()
            return x[((x - mu) / sd).abs() <= z_thresh].dropna()

        if outlier_mode == 'mad':
            med = np.nanmedian(v_clean)
            mad = np.nanmedian(np.abs(v_clean - med))
            if mad == 0 or not np.isfinite(mad):
                return x.dropna()
            c = 1.4826
            lo, hi = med - mad_k * c * mad, med + mad_k * c * mad
            return x[(x >= lo) & (x <= hi)].dropna()

        raise ValueError("outlier_mode must be 'none'|'iqr'|'zscore'|'mad'|'winsor'")

    def _robust_column_means(samples_df):
        out = {}
        for param in samples_df.columns:
            filt = _filter_or_winsorize(samples_df[param])
            if filt.size >= min_valid:
                out[param] = np.nanmedian(filt) if use_median_if_sparse else np.nanmean(filt)
            else:
                out[param] = (np.nanmedian(filt) if (use_median_if_sparse and filt.size > 0)
                              else np.nanmean(samples_df[param]))
        return pd.Series(out)

    all_results = {}
    for d in dirs:
        label = Path(d).name
        if verbose: print(f"\n=== Processing directory: {label} ===")

        sim_means_list, sim_names = [], []
        for i in sim_indices:
            sim_name = f"sim_{i}"
            if verbose: print(f"Processing {label}/{sim_name}...")
            try:
                data    = load_simulated_data(str(d), sim_name=sim_name)
                samples = approximator.sample(conditions=data, num_samples=num_samples)
                samples = workflow.samples_to_data_frame(samples)

                sim_means_list.append(_robust_column_means(samples))
                sim_names.append(sim_name)
            except (FileNotFoundError, ValueError) as e:
                if verbose: print(f"[{label}] Skipping {sim_name}: {e}")
            except Exception as e:
                if verbose: print(f"[{label}] Error in {sim_name}: {e}")

        if sim_means_list:
            df_sims = pd.concat(sim_means_list, axis=1)
            df_sims.columns = sim_names
            all_results[label] = df_sims
        elif verbose:
            print(f"[{label}] No simulation data was successfully processed.")

    if not all_results:
        if verbose: print("\nNo directories produced results.")
        return pd.DataFrame(), pd.DataFrame()

    mean_df = pd.DataFrame({label: df.mean(axis=1) for label, df in all_results.items()})
    if verbose:
        print("\nOverall Mean of Samples by directory (columns are dir names):")
        try:    display(mean_df)
        except NameError: print(mean_df.head())

    all_results_df = pd.concat(all_results, axis=0)
    all_results_df.index.names = ['directory', 'parameter']
    return all_results_df, mean_df


## 16. Bar Plot: Models vs Human (with significance stars)


In [ ]:
# Pretty-name lookup for any "<model>-<treatment>" directory key
DEFAULT_MODEL_NAMES = {
    #"llama2":   "Llama-2 7B",
    "kimi":     "Kimi K2",
    "llama70b": "Llama-3.3 70B",
    "llama3": "Llama-3.1 8B",
    "oss20b":   "GPT-OSS 20B",
    "oss120b":  "GPT-OSS 120B",
    "qwen3":     "Qwen-3.6-35B-A3B",
    "mistral":  "Mistral Small",
}

def _name_map_for_treatment(treatment):
    """Build {full_dir_key -> pretty_name} for a given suffix ('baseline','prompt','history')."""
    return {f"{k}-{treatment}": v for k, v in DEFAULT_MODEL_NAMES.items()}


@nature_style
def plot_dir_means_vs_human_with_sig(
    df_means,                 # rows=parameters, cols=directories (means)
    df_all_samples,           # MultiIndex (directory, parameter) x sim_* columns
    *,
    human_dir='human',
    err='sem',                # 'std' | 'sem' | 'ci'
    p_correction='BH',        # 'BH' | 'holm' | 'bonferroni' | 'none'
    correction_scope='per-parameter',  # 'per-parameter' | 'global'
    alpha=0.05,
    name_map=None,            # {full_dir_key -> pretty_name}; defaults to identity
    savepath=None,            # if None, no savefig
    return_table=True,
):
    """
    For each parameter:
      - Bar = per-directory mean across sims.
      - Error bar = std/sem/ci of per-sim values.
      - Dots = individual sessions.
      - Dashed line = human mean for that parameter.
      - Welch test vs human; BH/Holm/Bonferroni adjustment.
    """

    def _err_from(vals, mode):
        vals = np.asarray(vals, dtype=float)
        n = np.isfinite(vals).sum()
        if n == 0: return np.nan
        sd = np.nanstd(vals, ddof=1) if n > 1 else 0.0
        return {'std': sd, 'sem': sd/np.sqrt(n) if n else np.nan,
                'ci' : 1.96*sd/np.sqrt(n) if n else np.nan}[mode]

    def _adjust_pvals(pvals, method):
        pvals = np.asarray(pvals, float)
        out = np.full_like(pvals, np.nan, dtype=float)
        mask = np.isfinite(pvals); p = pvals[mask]; m = p.size
        if m == 0 or method.lower() == 'none':
            out[mask] = p;  return out
        ml = method.lower()
        if ml == 'bonferroni':
            out[mask] = np.minimum(p * m, 1.0); return out
        if ml == 'holm':
            order = np.argsort(p); adj = np.empty_like(p)
            for i, idx in enumerate(order): adj[idx] = (m - i) * p[idx]
            adj[order] = np.maximum.accumulate(np.minimum(adj[order], 1.0))
            out[mask] = np.minimum(adj, 1.0); return out
        if ml == 'bh':
            order = np.argsort(p); ranks = np.arange(1, m+1)
            bh_vals = (m / ranks) * p[order]
            bh_rev_cummin = np.minimum.accumulate(bh_vals[::-1])[::-1]
            adj = np.empty_like(p); adj[order] = np.minimum(bh_rev_cummin, 1.0)
            out[mask] = adj; return out
        raise ValueError("p_correction must be 'BH'|'holm'|'bonferroni'|'none'")

    def _stars(p):
        if not np.isfinite(p): return ""
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        return ""

    params    = sorted({idx[1] for idx in df_all_samples.index})
    dir_names = sorted({d for d, _ in df_all_samples.index if d != human_dir})

    means      = pd.DataFrame(index=params, columns=dir_names, dtype=float)
    errs       = pd.DataFrame(index=params, columns=dir_names, dtype=float)
    p_raw      = pd.DataFrame(index=params, columns=dir_names, dtype=float)
    n_x        = pd.DataFrame(index=params, columns=dir_names, dtype=float)
    n_y        = pd.Series(index=params, dtype=float)
    human_mean = pd.Series(index=params, dtype=float)

    for param in params:
        try:
            human_row = df_all_samples.loc[(human_dir, param)].astype(float)
        except KeyError:
            human_row = pd.Series(dtype=float)
        human_mean[param] = float(np.nanmean(human_row.values)) if len(human_row) else np.nan
        n_y[param]        = int(np.isfinite(human_row.values).sum()) if len(human_row) else 0

        for d in dir_names:
            try:
                row = df_all_samples.loc[(d, param)].astype(float)
            except KeyError:
                row = pd.Series(dtype=float)
            vals = row.values.astype(float)

            means.at[param, d] = float(np.nanmean(vals)) if vals.size else np.nan
            errs.at[param, d]  = _err_from(vals, err)
            n_x.at[param, d]   = int(np.isfinite(vals).sum())

            if len(human_row) and len(row):
                cols = row.index.intersection(human_row.index)
                xv = row.loc[cols].values.astype(float)
                yv = human_row.loc[cols].values.astype(float)
                m  = np.isfinite(xv) & np.isfinite(yv)
                xv, yv = xv[m], yv[m]
                p_raw.at[param, d] = (ttest_ind(xv, yv, equal_var=False, nan_policy='omit')[1]
                                     if xv.size >= 2 and yv.size >= 2 else np.nan)
            else:
                p_raw.at[param, d] = np.nan

    if correction_scope == 'global':
        flat = p_raw.values.flatten()
        adj_flat = _adjust_pvals(flat, p_correction)
        p_adj = pd.DataFrame(adj_flat.reshape(p_raw.shape), index=params, columns=dir_names)
    else:
        p_adj = p_raw.copy()
        for param in params:
            p_adj.loc[param, :] = _adjust_pvals(p_raw.loc[param, :].values, p_correction)

    # Figure: bar = mean across sessions ± s.e.m., dots = sessions, dashed line = Sterman mean;
    # stars = adjusted P (Welch t-test vs the Sterman-reported values).
    n_cols = 4
    n_rows = int(np.ceil(len(params) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(FULL_WIDTH, 47.5 * n_rows * MM), squeeze=False)
    axes = axes.ravel()
    name_map = name_map or {}

    for ax, param in zip(axes, params):
        rows = [df_all_samples.loc[(d, param)].astype(float).values for d in dir_names]
        all_values = np.concatenate(rows)
        set_dot_spacing(all_values, frac=0.02)
        star_y = np.nanmax(all_values) + (np.nanmax(all_values) - np.nanmin(all_values)) * 0.05
        for i, d in enumerate(dir_names):
            bar_with_dots(ax, i, rows[i], SET2_BARS[i % len(SET2_BARS)], 0.7, dot_size=1.2)
            label = _stars(p_adj.at[param, d])
            if label:
                ax.text(i, star_y, label, ha="center", va="bottom", fontsize=5)
        if np.isfinite(human_mean[param]):
            ax.axhline(human_mean[param], color="#555555", ls="--", lw=0.7, zorder=1)
        family, role = param.rsplit("_", 1)
        ax.set_title(f"{PARAM_LABELS.get(family, family)}, {ROLE_OF.get(role, role)}")
        ax.set_xticks(range(len(dir_names)))
        ax.set_xticklabels([name_map.get(d, d) for d in dir_names], rotation=40, ha="right", fontsize=5)

    for j in range(len(params), len(axes)):
        axes[j].axis("off")

    fig.legend(handles=[Line2D([], [], marker="o", ls="", color=DOT_COLOR, alpha=0.75, markeredgewidth=0,
                               markersize=2.5, label="Session ($n=11$)"),
                        Line2D([], [], color="#555555", ls="--", lw=0.7, label="Mean of Sterman-reported values")],
               loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
    fig.tight_layout(h_pad=0.8, w_pad=0.8)
    save_figure(fig, savepath)

    if not return_table:
        return fig

    rows = []
    for param in params:
        human_row = df_all_samples.loc[(human_dir, param)].astype(float)
        for d in dir_names:
            try:
                model_row = df_all_samples.loc[(d, param)].astype(float)
            except KeyError:
                model_row = pd.Series(dtype=float)
            cols = model_row.index.intersection(human_row.index)
            x = model_row.loc[cols].values.astype(float)
            y = human_row.loc[cols].values.astype(float)
            mask = np.isfinite(x) & np.isfinite(y)
            x, y = x[mask], y[mask]
            mae  = float(np.mean(np.abs(x - y))) if x.size else np.nan
            rmse = float(np.sqrt(np.mean((x - y)**2))) if x.size else np.nan
            rows.append({
                'parameter': param, 'model': d,
                'n_model': int(n_x.at[param, d]) if np.isfinite(n_x.at[param, d]) else 0,
                'n_human': int(n_y[param])     if np.isfinite(n_y[param])       else 0,
                'mean_model': means.at[param, d], 'mean_human': human_mean[param],
                'p_raw': p_raw.at[param, d], 'p_adj': p_adj.at[param, d],
                'MAE': mae, 'RMSE': rmse,
            })
    table_df = pd.DataFrame(rows).sort_values(['parameter', 'model']).reset_index(drop=True)
    return fig, table_df

## 17. Radar Plot: Roles vs Human (normalized to Human = 1)


In [ ]:
@nature_style
def plot_role_radars_vs_human(
    df_means,
    *,
    human_col='average human',
    fallback_human_cols=('human', 'Average Human', 'Human'),
    models=None,
    palette='Set2',
    name_map=None,
    savepath=None,
    title="Normalized to Human = 1 (Per-Role Radar Charts)",
    role_order=('Retailer','Distributor','Wholesaler','Factory'),
    role_suffix={'Retailer':'r','Distributor':'d','Wholesaler':'w','Factory':'f'},
    title_fs=14, label_fs=11, tick_fs=9, legend_fs=9,
    legend_loc='lower center', legend_ncol=4, label_pad=14,
    line_width=1.8, fill_alpha=0.08, marker='o', markersize=4,
):
    """Four polar subplots (one per role), each showing theta/alpha_s/beta/s_prime normalized to Human=1."""
    if human_col in df_means.columns:
        ref_col = human_col
    else:
        found = [c for c in fallback_human_cols if c in df_means.columns]
        if not found:
            raise ValueError(f"No human reference column found. Tried '{human_col}' and {fallback_human_cols}.")
        ref_col = found[0]

    if models is None:
        models = [c for c in df_means.columns if c != ref_col]
    else:
        models = [m for m in models if m in df_means.columns and m != ref_col]
    if not models:
        raise ValueError("No models to plot after excluding the human reference column.")

    pretty_param_labels = ["θ", "αS", "β", "S′"]
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), subplot_kw=dict(polar=True))
    axes = axes.ravel()

    name_map = name_map or {}
    display_names = [name_map.get(m, m) for m in models]
    cmap = mpl.colormaps.get_cmap(palette)
    base_colors = [cmap(i) for i in range(getattr(cmap, 'N', 8))]
    model_colors = list(itertools.islice(itertools.cycle(base_colors), len(models)))

    legend_handles, legend_labels = [], []
    for ax, role in zip(axes, role_order):
        sfx = role_suffix[role]
        role_rows = [f"theta_{sfx}", f"alpha_s_{sfx}", f"beta_{sfx}", f"s_prime_{sfx}"]
        missing = [r for r in role_rows if r not in df_means.index]
        if missing:
            ax.set_axis_off()
            ax.set_title(f"{role} (missing rows: {missing})", fontsize=label_fs)
            continue

        role_df  = df_means.loc[role_rows]
        role_norm = role_df.div(role_df[ref_col], axis=0)

        angles = np.linspace(0, 2*np.pi, len(role_rows), endpoint=False).tolist()
        angles += angles[:1]
        ax.set_theta_offset(np.pi / 2.0)
        ax.set_theta_direction(-1)

        for m, disp, color in zip(models, display_names, model_colors):
            values = role_norm[m].tolist(); values += values[:1]
            h, = ax.plot(angles, values, label=disp, linewidth=line_width,
                         marker=marker, markersize=markersize, color=color)
            ax.fill(angles, values, alpha=fill_alpha, color=h.get_color())
            if disp not in legend_labels:
                legend_handles.append(h); legend_labels.append(disp)

        ax.plot(angles, [1.0]*len(angles), linestyle="--", color="black", linewidth=1.2, label=ref_col)

        ax.set_thetagrids(np.degrees(angles[:-1]), pretty_param_labels)
        for lab in ax.get_xticklabels(): lab.set_fontsize(label_fs)
        ax.tick_params(axis='x', pad=label_pad)
        ax.tick_params(axis='y', labelsize=tick_fs, pad=2)
        for yt in ax.get_yticklabels(): yt.set_alpha(0.7)
        ax.grid(alpha=0.35); ax.spines['polar'].set_alpha(0.5)

        finite_vals = role_norm[models].to_numpy().ravel()
        finite_vals = finite_vals[np.isfinite(finite_vals)]
        if finite_vals.size:
            vmin, vmax = np.nanmin(finite_vals), np.nanmax(finite_vals)
            if vmin == vmax: vmin, vmax = vmin - 0.1, vmax + 0.1
            pad = 0.08 * (vmax - vmin)
            lower = min(vmin - pad, 1.0 - 0.15*(vmax - vmin))
            upper = max(vmax + pad, 1.0 + 0.15*(vmax - vmin))
            ax.set_ylim(max(0, lower), upper)

        ax.set_title(role, y=1.12, fontsize=title_fs-2)

    for j in range(len(role_order), len(axes)):
        axes[j].set_axis_off()
    fig.subplots_adjust(hspace=0.35, wspace=0.35)

    if legend_handles:
        fig.legend(legend_handles, legend_labels,
                   loc=legend_loc, bbox_to_anchor=(0.5, 0.0),
                   ncol=min(legend_ncol, len(legend_labels)),
                   frameon=False, prop={'size': legend_fs},
                   handlelength=2.2, columnspacing=1.2)

    if savepath:
        plt.savefig(savepath, format=str(savepath).split('.')[-1], dpi=350)
    plt.show()

## 18. Agreement Metrics (MAE / RMSE / Pearson / Spearman)


In [ ]:
def _align_xy(df_all_samples, model_dir, human_dir, param):
    """Aligned (x_model, y_human) over intersecting sim columns, finite-only."""
    m = df_all_samples.loc[(model_dir, param)]
    h = df_all_samples.loc[(human_dir, param)]
    cols = sorted(set(m.index).intersection(set(h.index)))
    x = m[cols].astype(float).values
    y = h[cols].astype(float).values
    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]


def _safe_pearson(x, y):
    try:
        res = pearsonr(x, y)
        return res.statistic if hasattr(res, "statistic") else res[0]
    except Exception:
        return np.nan


def _safe_spearman(x, y):
    try:    return spearmanr(x, y).correlation
    except Exception: return np.nan


def compute_basic_agreement_for_model(df_all_samples, model_dir,
                                       human_dir="human", *, do_spearman=True):
    """Per-parameter and per-role MAE/RMSE/Pearson_r [/Spearman_rho]."""
    params = sorted({idx[1] for idx in df_all_samples.index})

    rows = []
    for p in params:
        if (model_dir, p) not in df_all_samples.index or (human_dir, p) not in df_all_samples.index:
            continue
        x, y = _align_xy(df_all_samples, model_dir, human_dir, p)
        if len(x) < 2:
            rows.append({'parameter': p, 'n_pairs': len(x), 'MAE': np.nan, 'RMSE': np.nan,
                         'Pearson_r': np.nan, **({'Spearman_rho': np.nan} if do_spearman else {})})
            continue
        out = {'parameter': p, 'n_pairs': len(x),
               'MAE': float(np.mean(np.abs(x - y))),
               'RMSE': float(np.sqrt(np.mean((x - y)**2))),
               'Pearson_r': _safe_pearson(x, y)}
        if do_spearman: out['Spearman_rho'] = _safe_spearman(x, y)
        rows.append(out)
    param_metrics_df = pd.DataFrame(rows).set_index('parameter').sort_index()

    role_map = {'Retailer':'r','Distributor':'d','Wholesaler':'w','Factory':'f'}
    role_rows = []
    for role, sfx in role_map.items():
        pnames = [f"theta_{sfx}", f"alpha_s_{sfx}", f"beta_{sfx}", f"s_prime_{sfx}"]
        if any(((model_dir, p) not in df_all_samples.index or (human_dir, p) not in df_all_samples.index)
               for p in pnames):
            role_rows.append({'role': role, 'n_pairs': 0, 'MAE': np.nan, 'RMSE': np.nan,
                              'Pearson_r': np.nan, **({'Spearman_rho': np.nan} if do_spearman else {})})
            continue

        cols = None
        for p in pnames:
            mc = set(df_all_samples.loc[(model_dir, p)].index)
            hc = set(df_all_samples.loc[(human_dir, p)].index)
            cols = (mc & hc) if cols is None else (cols & mc & hc)
        cols = sorted(cols) if cols else []
        if not cols:
            role_rows.append({'role': role, 'n_pairs': 0, 'MAE': np.nan, 'RMSE': np.nan,
                              'Pearson_r': np.nan, **({'Spearman_rho': np.nan} if do_spearman else {})})
            continue

        x_list, y_list = [], []
        for c in cols:
            for p in pnames:
                xv = df_all_samples.loc[(model_dir, p), c]
                yv = df_all_samples.loc[(human_dir, p), c]
                if np.isfinite(xv) and np.isfinite(yv):
                    x_list.append(float(xv)); y_list.append(float(yv))

        x = np.array(x_list); y = np.array(y_list)
        if len(x) < 2:
            role_rows.append({'role': role, 'n_pairs': len(x), 'MAE': np.nan, 'RMSE': np.nan,
                              'Pearson_r': np.nan, **({'Spearman_rho': np.nan} if do_spearman else {})})
            continue
        out = {'role': role, 'n_pairs': len(x),
               'MAE': float(np.mean(np.abs(x - y))),
               'RMSE': float(np.sqrt(np.mean((x - y)**2))),
               'Pearson_r': _safe_pearson(x, y)}
        if do_spearman: out['Spearman_rho'] = _safe_spearman(x, y)
        role_rows.append(out)

    role_metrics_df = pd.DataFrame(role_rows).set_index('role')
    return param_metrics_df, role_metrics_df


def compute_basic_agreement_for_all_models(df_all_samples, human_dir="human", *, do_spearman=True):
    """Same as `compute_basic_agreement_for_model` but stacked over all model dirs."""
    model_dirs = sorted({d for d, _ in df_all_samples.index})
    model_dirs = [m for m in model_dirs if m != human_dir]

    p_rows, r_rows = [], []
    for m in model_dirs:
        pm, rm = compute_basic_agreement_for_model(df_all_samples, m, human_dir, do_spearman=do_spearman)
        pm = pm.copy(); pm['model'] = m; p_rows.append(pm.reset_index())
        rm = rm.copy(); rm['model'] = m; r_rows.append(rm.reset_index())

    all_param = (pd.concat(p_rows, ignore_index=True)
                   .set_index(['model','parameter']).sort_index())
    all_role  = (pd.concat(r_rows, ignore_index=True)
                   .set_index(['model','role']).sort_index())
    return all_param, all_role


## 19. Per-Experiment Configuration

Each LLM is run under three treatments:
- **baseline**: no special prompting
- **prompt**: heuristic-aware prompt
- **history**: with conversation history (few-shot)

Each directory holds 11 `sim_*_extracted.json` files.


In [ ]:
MODEL_NAMES_FULL = ["llama70b", "kimi", "oss20b", "oss120b",
                    "qwen3", "llama3", "mistral"]

def dirs_for(treatment):
    return [str(DATA_ROOT / f"{m}-{treatment}") for m in MODEL_NAMES_FULL]

DIRS_BASELINE = dirs_for("baseline")
DIRS_PROMPT   = dirs_for("prompt")
DIRS_HISTORY  = dirs_for("alt")


In [ ]:
# ---------------------------------------------------------------------
# Reviewer check 3:
# Posterior predictive checks for selected LLM trajectories
# ---------------------------------------------------------------------

# Pre-specified representative cases. Using the same sim index across
# treatments avoids selecting a different session after inspecting results.
LLM_PPC_CASES = [
    {
        "label": "Llama3.1 8B — baseline — sim_0",
        "directory": (
            Path(DATA_ROOT)
            / "llama3-baseline"
        ),
        "sim_name": "sim_0",
    },
    {
        "label": "Llama3.1 8B — heuristic-aware — sim_0",
        "directory": (
            Path(DATA_ROOT)
            / "llama3-prompt"
        ),
        "sim_name": "sim_0",
    },
    {
        "label": "Qwen-3.6-35B-A3B — baseline — sim_0",
        "directory": (
            Path(DATA_ROOT)
            / "qwen3-baseline"
        ),
        "sim_name": "sim_0",
    },
    {
        "label": "Qwen-3.6-35B-A3B — heuristic-aware — sim_0",
        "directory": (
            Path(DATA_ROOT)
            / "qwen3-prompt"
        ),
        "sim_name": "sim_0",
    },
]

LLM_PPC_POSTERIOR_SAMPLES = 5000
LLM_PPC_DRAWS = 1000

llm_ppc_metrics = []
llm_ppc_posteriors = {}
llm_ppc_observed = {}

with PdfPages(
    PLOT_DIR
    / "ppc_llm_selected_sessions.pdf"
) as pdf:

    for case_id, case in enumerate(
        LLM_PPC_CASES
    ):
        if not case["directory"].exists():
            raise FileNotFoundError(
                "LLM PPC directory does not exist: "
                f"{case['directory']}"
            )

        observed = load_simulated_data(
            str(case["directory"]),
            sim_name=case["sim_name"],
        )

        posterior = infer_posterior_frame(
            observed,
            num_samples=(
                LLM_PPC_POSTERIOR_SAMPLES
            ),
            seed=60_000 + case_id,
        )

        predictive = draw_posterior_predictive(
            posterior,
            n_draws=LLM_PPC_DRAWS,
            seed=70_000 + case_id,
            sigma_eps=1.0,
            simulator_seed=42,
            vary_simulator_seed=False,
        )

        llm_ppc_observed[
            case["label"]
        ] = observed

        llm_ppc_posteriors[
            case["label"]
        ] = posterior

        metrics = summarize_posterior_predictive(
            observed,
            predictive,
            label=case["label"],
            interval=0.90,
        )
        metrics.insert(
            0,
            "source",
            "LLM",
        )
        llm_ppc_metrics.append(
            metrics
        )

        figure = make_order_ppc_figure(
            observed,
            predictive,
            title=(
                "LLM posterior predictive check: "
                f"{case['label']}"
            ),
        )

        # One multipage PDF with all selected LLM cases.
        pdf.savefig(
            figure,
            bbox_inches="tight",
        )

        # One individual PDF per selected case.
        figure.savefig(
            PLOT_DIR
            / f"ppc_llm_case_{case_id + 1}.pdf",
            bbox_inches="tight",
        )

        plt.close(figure)

llm_ppc_metrics = pd.concat(
    llm_ppc_metrics,
    ignore_index=True,
)

llm_ppc_metrics.to_csv(
    PLOT_DIR
    / "ppc_llm_selected_metrics.csv",
    index=False,
)

# The order-only table directly addresses whether each LLM trajectory can
# be represented by the Sterman ABM class.
llm_order_ppc_summary = (
    llm_ppc_metrics.loc[
        llm_ppc_metrics[
            "variable"
        ] == "order"
    ]
    .groupby("label")[
        [
            "pointwise_coverage",
            "rmse_to_predictive_median",
            "mae_to_predictive_median",
            "mean_interval_width",
        ]
    ]
    .mean()
    .sort_index()
)

display(
    llm_order_ppc_summary.round(3)
)

## 20. Run: Baseline Experiment


In [ ]:
keras.utils.set_random_seed(SEED)

all_samples_by_dir, overall_means_by_dir = compute_means_for_dirs(
    DIRS_BASELINE, sim_indices=range(11), verbose=True)
all_samples_by_dir, overall_means_by_dir = append_humans_to_outputs(
    all_samples_by_dir, overall_means_by_dir)

# Per-parameter FDR-corrected bar plot
fig_baseline, tbl_baseline = plot_dir_means_vs_human_with_sig(
    overall_means_by_dir, all_samples_by_dir,
    human_dir='human', err='sem',
    p_correction='BH', correction_scope='per-parameter',
    name_map=_name_map_for_treatment("baseline"),
    savepath=PLOT_DIR / "parameter_baseline.pdf",
    return_table=True,
)

# Per-model agreement table (RMSE/MAE/r/rho)
param_metrics_baseline, role_metrics_baseline = compute_basic_agreement_for_model(
    all_samples_by_dir, model_dir='kimi-baseline', human_dir='human', do_spearman=True)
print(param_metrics_baseline.round(3))
print(role_metrics_baseline.round(3))

all_param_baseline, all_role_baseline = compute_basic_agreement_for_all_models(
    all_samples_by_dir, human_dir='human', do_spearman=True)
print(all_role_baseline.groupby('model')[['RMSE','MAE','Pearson_r','Spearman_rho']].mean().round(3))

# Radar
plot_role_radars_vs_human(
    overall_means_by_dir, human_col='human',
    palette="Set2_cvd",   # Set2, colour-blind repaired
    name_map=_name_map_for_treatment("baseline"),
    savepath=PLOT_DIR / "radars_baseline.pdf",
)

## 21. Run: Prompt Experiment


In [ ]:
keras.utils.set_random_seed(SEED)

all_samples_prompt_by_dir, overall_means_prompt_by_dir = compute_means_for_dirs(
    DIRS_PROMPT, sim_indices=range(11), verbose=True)
all_samples_prompt_by_dir, overall_means_prompt_by_dir = append_humans_to_outputs(
    all_samples_prompt_by_dir, overall_means_prompt_by_dir)
display(overall_means_prompt_by_dir)

fig_prompt, tbl_prompt = plot_dir_means_vs_human_with_sig(
    overall_means_prompt_by_dir, all_samples_prompt_by_dir,
    human_dir='human', err='sem',
    p_correction='BH', correction_scope='per-parameter',
    name_map=_name_map_for_treatment("prompt"),
    savepath=PLOT_DIR / "parameter_prompt.pdf",
    return_table=True,
)

all_param_prompt, all_role_prompt = compute_basic_agreement_for_all_models(
    all_samples_prompt_by_dir, human_dir='human', do_spearman=True)
print(all_role_prompt.groupby('model')[['RMSE','MAE','Pearson_r','Spearman_rho']].mean().round(3))

plot_role_radars_vs_human(
    overall_means_prompt_by_dir, human_col='human',
    palette="Set2_cvd",   # Set2, colour-blind repaired
    name_map=_name_map_for_treatment("prompt"),
    savepath=PLOT_DIR / "radars_prompt.pdf",
)

## 22. Run: Alt Experiment


In [ ]:
keras.utils.set_random_seed(SEED)

all_samples_few_by_dir, overall_means_few_by_dir = compute_means_for_dirs(
    DIRS_HISTORY, sim_indices=range(11), verbose=False)
all_samples_few_by_dir, overall_means_few_by_dir = append_humans_to_outputs(
    all_samples_few_by_dir, overall_means_few_by_dir)
display(overall_means_few_by_dir)

fig_few, tbl_few = plot_dir_means_vs_human_with_sig(
    overall_means_few_by_dir, all_samples_few_by_dir,
    human_dir='human', err='sem',
    p_correction='BH', correction_scope='per-parameter',
    name_map=_name_map_for_treatment("alt"),
    savepath=PLOT_DIR / "parameter_alt.pdf",
    return_table=True,
)

plot_role_radars_vs_human(
    overall_means_few_by_dir, human_col='human',
    palette="Set2_cvd",   # Set2, colour-blind repaired
    name_map=_name_map_for_treatment("alt"),
    savepath=PLOT_DIR / "radars_alt.pdf",
)

## 23. Overlay: Baseline vs Prompt vs Human (per parameter)


In [ ]:
def plot_overlay_baseline_prompt_vs_human(
    df_means_base, df_all_samples_base,
    df_means_prompt, df_all_samples_prompt,
    *,
    human_dir='human',
    err='sem',
    p_correction='BH',
    correction_scope='per-parameter',
    alpha=0.05,
    title="Baseline vs Prompt overlaid (Welch t-test + FDR)",
    name_map=None,
    savepath=None,
    return_table=True,
):
    """
    Overlay bar plots for baseline and prompt scenarios on the same axes.
    Models are paired by 'normalized key' (column with '-baseline'/'-prompt' stripped).
    """

    def _err_from(vals, mode):
        vals = np.asarray(vals, dtype=float)
        n = np.isfinite(vals).sum()
        if n == 0: return np.nan
        sd = np.nanstd(vals, ddof=1) if n > 1 else 0.0
        return {'std': sd, 'sem': sd/np.sqrt(n) if n else np.nan,
                'ci' : 1.96*sd/np.sqrt(n) if n else np.nan}[mode]

    def _adjust_pvals(pvals, method):
        pvals = np.asarray(pvals, float)
        out = np.full_like(pvals, np.nan, dtype=float)
        mask = np.isfinite(pvals); p = pvals[mask]; m = p.size
        if m == 0 or method.lower() == 'none':
            out[mask] = p; return out
        ml = method.lower()
        if ml == 'bonferroni': out[mask] = np.minimum(p * m, 1.0); return out
        if ml == 'holm':
            order = np.argsort(p); adj = np.empty_like(p)
            for i, idx in enumerate(order): adj[idx] = (m - i) * p[idx]
            adj[order] = np.maximum.accumulate(np.minimum(adj[order], 1.0))
            out[mask] = np.minimum(adj, 1.0); return out
        if ml == 'bh':
            order = np.argsort(p); ranks = np.arange(1, m+1)
            bh_vals = (m / ranks) * p[order]
            bh_rev_cummin = np.minimum.accumulate(bh_vals[::-1])[::-1]
            adj = np.empty_like(p); adj[order] = np.minimum(bh_rev_cummin, 1.0)
            out[mask] = adj; return out
        raise ValueError("p_correction must be 'BH'|'holm'|'bonferroni'|'none'")

    def _stars(p):
        if not np.isfinite(p): return ""
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        return ""

    def _norm_key(col):
        if col.endswith("-baseline"): return col[:-9]
        if col.endswith("-prompt"):   return col[:-7]
        return col

    def _lighten(color, factor=0.55):
        r, g, b = to_rgb(color)
        return (1 - factor*(1-r), 1 - factor*(1-g), 1 - factor*(1-b))

    params = sorted({idx[1] for idx in df_all_samples_base.index})
    base_cols   = [c for c in df_means_base.columns   if c != human_dir]
    prompt_cols = [c for c in df_means_prompt.columns if c != human_dir]
    base_map    = {_norm_key(c): c for c in base_cols}
    prompt_map  = {_norm_key(c): c for c in prompt_cols}
    model_keys  = [k for k in base_map if k in prompt_map]
    if name_map is not None:
        allowed = set(name_map.keys())
        model_keys = [k for k in model_keys if k in allowed]
    if not model_keys:
        raise ValueError("No overlapping model keys between baseline and prompt.")
    if name_map is None:
        name_map = {k: k for k in model_keys}
    disp_names = [name_map[k] for k in model_keys]

    def _compute(df_all, df_means, src_map):
        means = pd.DataFrame(index=params, columns=model_keys, dtype=float)
        errs  = pd.DataFrame(index=params, columns=model_keys, dtype=float)
        p_raw = pd.DataFrame(index=params, columns=model_keys, dtype=float)
        for param in params:
            try: human_row = df_all.loc[(human_dir, param)].astype(float)
            except KeyError: human_row = pd.Series(dtype=float)
            for k in model_keys:
                col = src_map[k]
                try: row = df_all.loc[(col, param)].astype(float)
                except KeyError: row = pd.Series(dtype=float)
                vals = row.values.astype(float)
                means.at[param, k] = float(np.nanmean(vals)) if vals.size else np.nan
                errs.at[param, k]  = _err_from(vals, err)
                if len(human_row) and len(row):
                    cols = row.index.intersection(human_row.index)
                    xv = row.loc[cols].values.astype(float)
                    yv = human_row.loc[cols].values.astype(float)
                    m = np.isfinite(xv) & np.isfinite(yv)
                    xv, yv = xv[m], yv[m]
                    p_raw.at[param, k] = (ttest_ind(xv, yv, equal_var=False, nan_policy='omit')[1]
                                          if xv.size >= 2 and yv.size >= 2 else np.nan)
                else:
                    p_raw.at[param, k] = np.nan
        p_adj = p_raw.copy()
        if correction_scope == 'global':
            flat = _adjust_pvals(p_raw.values.flatten(), p_correction)
            p_adj = pd.DataFrame(flat.reshape(p_raw.shape), index=params, columns=model_keys)
        else:
            for param in params:
                p_adj.loc[param, :] = _adjust_pvals(p_raw.loc[param, :].values, p_correction)
        return means, errs, p_raw, p_adj

    means_base,   errs_base,   p_raw_base,   p_adj_base   = _compute(df_all_samples_base,   df_means_base,   base_map)
    means_prompt, errs_prompt, p_raw_prompt, p_adj_prompt = _compute(df_all_samples_prompt, df_means_prompt, prompt_map)

    human_mean = pd.Series(index=params, dtype=float)
    for param in params:
        try: hr = df_all_samples_base.loc[(human_dir, param)].astype(float)
        except KeyError: hr = pd.Series(dtype=float)
        human_mean[param] = float(np.nanmean(hr.values)) if len(hr) else np.nan

    n_params = len(params)
    n_cols = 4; n_rows = int(np.ceil(n_params / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 6.0*n_rows), squeeze=False)
    axes = axes.ravel()

    cmap = cm.get_cmap("Set2", max(3, len(model_keys)))
    prompt_colors   = [cmap(i) for i in range(len(model_keys))]
    baseline_colors = [_lighten(c, 0.70) for c in prompt_colors]
    base_width, prompt_width = 0.82, 0.62
    base_alpha,  prompt_alpha = 0.45, 0.95

    for idx, param in enumerate(params):
        ax = axes[idx]
        x  = np.arange(len(model_keys))
        yb = means_base.loc[param].values.astype(float)
        eb = errs_base.loc[param].values.astype(float)
        yp = means_prompt.loc[param].values.astype(float)
        ep = errs_prompt.loc[param].values.astype(float)
        pp = p_adj_prompt.loc[param].values.astype(float)

        ax.bar(x, yb, color=baseline_colors, alpha=base_alpha, width=base_width,
               ecolor='black', linewidth=0.6, edgecolor='black', hatch='//',
               zorder=2, label="Baseline" if idx == 0 else None)
        ax.bar(x, yp, yerr=ep, capsize=4, color=prompt_colors, alpha=prompt_alpha,
               width=prompt_width, ecolor='black', linewidth=0.6,
               edgecolor='black', zorder=3, label="Prompt" if idx == 0 else None)

        if np.isfinite(human_mean[param]):
            ax.axhline(human_mean[param], color='gray', linestyle='--',
                       linewidth=1, alpha=0.85, zorder=1,
                       label="Human mean" if idx == 0 else None)

        for xi, yj, ej, pval in zip(x, yp, ep, pp):
            lab = _stars(pval)
            if lab:
                v_off = (ej if np.isfinite(ej) else np.nanmax([*eb, *ep, 0])) * 0.6
                ax.annotate(lab, xy=(xi, yj + v_off), xytext=(0, 0),
                            textcoords='offset points', ha='center', va='bottom',
                            fontsize=11, fontweight='bold', zorder=4)

        ax.set_title(param)
        ax.set_xticks(x)
        ax.set_xticklabels(disp_names, rotation=35, ha='right')
        ax.set_ylabel("Value")

    for j in range(n_params, len(axes)):
        axes[j].axis('off')

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False,
                   bbox_to_anchor=(0.5, 0.0))
    fig.tight_layout()
    plt.show()

    if not return_table:
        return fig

    rows = []
    for param in params:
        hrow = df_all_samples_base.loc[(human_dir, param)].astype(float)
        for k in model_keys:
            bcol, pcol = base_map[k], prompt_map[k]
            try: xb = df_all_samples_base.loc[(bcol, param)].astype(float)
            except KeyError: xb = pd.Series(dtype=float)
            cols = xb.index.intersection(hrow.index)
            xbv = xb.loc[cols].values.astype(float); yv = hrow.loc[cols].values.astype(float)
            m   = np.isfinite(xbv) & np.isfinite(yv)
            xbv, yv2 = xbv[m], yv[m]
            mae_b  = float(np.mean(np.abs(xbv - yv2))) if xbv.size else np.nan
            rmse_b = float(np.sqrt(np.mean((xbv - yv2)**2))) if xbv.size else np.nan
            try: xp = df_all_samples_prompt.loc[(pcol, param)].astype(float)
            except KeyError: xp = pd.Series(dtype=float)
            cols = xp.index.intersection(hrow.index)
            xpv = xp.loc[cols].values.astype(float); yv = hrow.loc[cols].values.astype(float)
            m   = np.isfinite(xpv) & np.isfinite(yv)
            xpv, yv3 = xpv[m], yv[m]
            mae_p  = float(np.mean(np.abs(xpv - yv3))) if xpv.size else np.nan
            rmse_p = float(np.sqrt(np.mean((xpv - yv3)**2))) if xpv.size else np.nan

            rows.append({
                'parameter': param, 'model_key': k,
                'model_display': name_map.get(k, k),
                'mean_baseline': means_base.at[param, k],
                'err_baseline':  errs_base.at[param, k],
                'mean_prompt':   means_prompt.at[param, k],
                'err_prompt':    errs_prompt.at[param, k],
                'p_raw_baseline': p_raw_base.at[param, k],
                'p_adj_baseline': p_adj_base.at[param, k],
                'p_raw_prompt':   p_raw_prompt.at[param, k],
                'p_adj_prompt':   p_adj_prompt.at[param, k],
                'MAE_baseline': mae_b, 'RMSE_baseline': rmse_b,
                'MAE_prompt':   mae_p, 'RMSE_prompt':   rmse_p,
            })
    table_df = pd.DataFrame(rows).sort_values(['parameter','model_display']).reset_index(drop=True)
    return fig, table_df


# --- Run overlay plot ---
OVERLAY_NAME_MAP = {
    "llama2":   "Llama-2 7B",
    "kimi":     "Kimi K2",
    "llama3": "Llama-3.1 8B",
    "llama70b": "Llama-3.3 70B",
    "oss20b":   "GPT-OSS 20B",
    "oss120b":  "GPT-OSS 120B",
    "qwen3":     "Qwen-3.6-35B-A3B",
    "mistral":  "Mistral Small",
    # commented out in original: gpt20b, gpt120b, qwen7b
}

fig_overlay, tbl_overlay = plot_overlay_baseline_prompt_vs_human(
    overall_means_by_dir,        all_samples_by_dir,
    overall_means_prompt_by_dir, all_samples_prompt_by_dir,
    human_dir='human', err='sem',
    p_correction='BH', correction_scope='per-parameter',
    name_map=OVERLAY_NAME_MAP,
    savepath=PLOT_DIR / "parameter_overlay.pdf",
    return_table=True,
)


## 24. Overlay: Collapsed by Role (4 panels, normalized to Human = 1)


In [ ]:
def plot_overlay_roles_collapsed(
    df_means_base, df_all_samples_base,
    df_means_prompt, df_all_samples_prompt,
    *,
    human_dir='human',
    err='sem',
    p_correction='BH',
    correction_scope='per-parameter',
    alpha=0.05,
    title="Baseline vs Prompt - Collapsed by Role (normalized to Human=1)",
    name_map=None,
    savepath=None,
    return_table=True,
):
    """4 subplots (one per role) - bars are per-model means of theta/alpha_s/beta/s_prime normalized to human."""

    def _err_from(vals, mode):
        vals = np.asarray(vals, float); vals = vals[np.isfinite(vals)]
        n = vals.size
        if n == 0: return np.nan
        sd = np.nanstd(vals, ddof=1) if n > 1 else 0.0
        return {'std': sd, 'sem': sd/np.sqrt(n) if n else np.nan,
                'ci' : 1.96*sd/np.sqrt(n) if n else np.nan}[mode]

    def _adjust_pvals(pvals, method):
        pvals = np.asarray(pvals, float)
        out = np.full_like(pvals, np.nan, float)
        mask = np.isfinite(pvals); p = pvals[mask]; m = p.size
        if m == 0 or method.lower() == 'none':
            out[mask] = p; return out
        ml = method.lower()
        if ml == 'bonferroni': out[mask] = np.minimum(p * m, 1.0); return out
        if ml == 'holm':
            order = np.argsort(p); adj = np.empty_like(p)
            for i, idx in enumerate(order): adj[idx] = (m - i) * p[idx]
            adj[order] = np.maximum.accumulate(np.minimum(adj[order], 1.0))
            out[mask] = np.minimum(adj, 1.0); return out
        if ml == 'bh':
            order = np.argsort(p); ranks = np.arange(1, m+1)
            bh_vals = (m / ranks) * p[order]
            bh_rev_cummin = np.minimum.accumulate(bh_vals[::-1])[::-1]
            adj = np.empty_like(p); adj[order] = np.minimum(bh_rev_cummin, 1.0)
            out[mask] = adj; return out
        raise ValueError

    def _stars(p):
        if not np.isfinite(p): return ""
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        return ""

    def _norm_key(col):
        if col.endswith("-baseline"): return col[:-9]
        if col.endswith("-prompt"):   return col[:-7]
        return col

    def _lighten(color, factor=0.55):
        r, g, b = to_rgb(color)
        return (1 - factor*(1-r), 1 - factor*(1-g), 1 - factor*(1-b))

    role_suffix = {'Retailer':'r','Distributor':'d','Wholesaler':'w','Factory':'f'}
    role_params = {role: [f"theta_{s}", f"alpha_s_{s}", f"beta_{s}", f"s_prime_{s}"]
                   for role, s in role_suffix.items()}

    base_cols   = [c for c in df_means_base.columns   if c != human_dir]
    prompt_cols = [c for c in df_means_prompt.columns if c != human_dir]
    base_map    = {_norm_key(c): c for c in base_cols}
    prompt_map  = {_norm_key(c): c for c in prompt_cols}
    model_keys  = [k for k in base_map if k in prompt_map]
    if name_map is not None:
        allowed = set(name_map.keys())
        model_keys = [k for k in model_keys if k in allowed]
    if not model_keys:
        raise ValueError("No overlapping model keys.")
    if name_map is None:
        name_map = {k: k for k in model_keys}
    disp_names = [name_map[k] for k in model_keys]

    human_means = {}
    for role, rows in role_params.items():
        for p in rows:
            try: hr = df_all_samples_base.loc[(human_dir, p)].astype(float)
            except KeyError: hr = pd.Series(dtype=float)
            human_means[p] = float(np.nanmean(hr.values)) if len(hr) else np.nan

    def _norm_score_per_sim(df_all, col_map, param_list):
        out = {k: [] for k in model_keys}
        out['human'] = []
        for k in list(model_keys) + ['human']:
            col = col_map.get(k, k) if k != 'human' else human_dir
            ns = []
            for p in param_list:
                try:
                    s = df_all.loc[(col, p)].astype(float)
                    s = s[np.isfinite(s)]
                    denom = human_means[p]
                    if np.isfinite(denom) and denom != 0:
                        ns.append(s / denom)
                except KeyError:
                    pass
            if not ns: out[k] = np.array([]); continue
            common = ns[0].index
            for s in ns[1:]: common = common.intersection(s.index)
            out[k] = np.array([]) if len(common) == 0 else np.nanmean(
                np.vstack([s.loc[common].values for s in ns]), axis=0)
        return out

    fig, axes = plt.subplots(1, 4, figsize=(22, 5.5), sharey=False)
    cmap = cm.get_cmap("Set2", max(3, len(model_keys)))
    prompt_colors   = [cmap(i) for i in range(len(model_keys))]
    baseline_colors = [_lighten(c, 0.70) for c in prompt_colors]

    rows_tidy = []
    for ax, (role, plist) in zip(axes, role_params.items()):
        scores_b = _norm_score_per_sim(df_all_samples_base,   base_map,   plist)
        scores_p = _norm_score_per_sim(df_all_samples_prompt, prompt_map, plist)
        h_scores = scores_b['human']

        mb, eb_, mp, ep_, pb, pp = [], [], [], [], [], []
        for k in model_keys:
            xb = scores_b[k]; xp = scores_p[k]
            mb.append(np.nanmean(xb) if xb.size else np.nan)
            eb_.append(_err_from(xb, err))
            mp.append(np.nanmean(xp) if xp.size else np.nan)
            ep_.append(_err_from(xp, err))
            pb.append(ttest_ind(xb, h_scores, equal_var=False, nan_policy='omit')[1]
                      if xb.size >= 2 and h_scores.size >= 2 else np.nan)
            pp.append(ttest_ind(xp, h_scores, equal_var=False, nan_policy='omit')[1]
                      if xp.size >= 2 and h_scores.size >= 2 else np.nan)

        pb_adj = _adjust_pvals(np.array(pb), p_correction)
        pp_adj = _adjust_pvals(np.array(pp), p_correction)

        x = np.arange(len(model_keys))
        ax.bar(x, mb, color=baseline_colors, alpha=0.45, width=0.82,
               linewidth=0.6, edgecolor='black', hatch='//', zorder=2,
               label="Baseline" if role == "Retailer" else None)
        ax.bar(x, mp, yerr=ep_, capsize=4, color=prompt_colors, alpha=0.95,
               width=0.62, ecolor='black', linewidth=0.6, edgecolor='black',
               zorder=3, label="Prompt" if role == "Retailer" else None)
        ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.9,
                   label="Human mean" if role == "Retailer" else None, zorder=1)

        for xi, yj, ej, pval in zip(x, mp, ep_, pp_adj):
            lab = _stars(pval)
            if lab:
                v_off = (ej if np.isfinite(ej) else 0.05) * 0.6
                ax.text(xi, yj + v_off, lab, ha='center', va='bottom',
                        fontsize=11, fontweight='bold', zorder=4)

        ax.set_title(role)
        ax.set_xticks(x); ax.set_xticklabels(disp_names, rotation=35, ha='right')
        if role == "Retailer":
            ax.set_ylabel("Normalized parameter value (mean across theta, alpha_s, beta, s_prime)")

        for i, k in enumerate(model_keys):
            rows_tidy.append({
                'role': role, 'model_key': k, 'model_display': name_map.get(k, k),
                'mean_baseline_norm': mb[i], 'err_baseline_norm': eb_[i],
                'mean_prompt_norm':   mp[i], 'err_prompt_norm':   ep_[i],
                'p_raw_baseline': pb[i], 'p_adj_baseline': pb_adj[i],
                'p_raw_prompt':   pp[i], 'p_adj_prompt':   pp_adj[i],
            })

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='lower center', ncol=3,
                   frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, format=str(savepath).split('.')[-1], dpi=350,
                    bbox_inches="tight", pad_inches=0.2)
    plt.show()

    if not return_table:
        return fig
    return fig, pd.DataFrame(rows_tidy)


fig_roles, tbl_roles = plot_overlay_roles_collapsed(
    overall_means_by_dir,        all_samples_by_dir,
    overall_means_prompt_by_dir, all_samples_prompt_by_dir,
    human_dir='human', err='sem',
    p_correction='BH', correction_scope='per-parameter',
    name_map=OVERLAY_NAME_MAP,
    savepath=PLOT_DIR / "parameter_overlay_roles.pdf",
    return_table=True,
)


## 25. Overlay: Aggregated Across Roles (with robust normalization)


In [ ]:
@nature_style
def plot_overlay_aggregated(
    df_means_base, df_all_samples_base,
    df_means_prompt, df_all_samples_prompt,
    *,
    human_dir='human',
    err='sem',
    p_correction='BH',
    alpha=0.05,
    robust_method='zscore',           # 'none' | 'zscore' | 'winsorize' | 'median'
    winsor_limits=(0.05, 0.95),
    name_map=None,
    order=None,                       # optional list of *display names* for x-axis order
    savepath=None,
    return_table=True,
):
    """
    Single bar plot, one bar per (model x treatment), where the height is the per-sim
    score averaged across all 16 parameters (or 4 roles x 4 params).
    `robust_method` controls how each parameter's draws are normalized before averaging.
    """

    def _err_from(vals, mode):
        vals = np.asarray(vals, float); vals = vals[np.isfinite(vals)]
        n = vals.size
        if n == 0: return np.nan
        sd = np.nanstd(vals, ddof=1) if n > 1 else 0.0
        return {'std': sd, 'sem': sd/np.sqrt(n) if n else np.nan,
                'ci' : 1.96*sd/np.sqrt(n) if n else np.nan}[mode]

    def _adjust_pvals(pvals, method):
        pvals = np.asarray(pvals, float)
        out = np.full_like(pvals, np.nan, float)
        mask = np.isfinite(pvals); p = pvals[mask]; m = p.size
        if m == 0 or method.lower() == 'none':
            out[mask] = p; return out
        ml = method.lower()
        if ml == 'bonferroni': out[mask] = np.minimum(p * m, 1.0); return out
        if ml == 'holm':
            order_ = np.argsort(p); adj = np.empty_like(p)
            for i, idx in enumerate(order_): adj[idx] = (m - i) * p[idx]
            adj[order_] = np.maximum.accumulate(np.minimum(adj[order_], 1.0))
            out[mask] = np.minimum(adj, 1.0); return out
        if ml == 'bh':
            order_ = np.argsort(p); ranks = np.arange(1, m+1)
            bh_vals = (m / ranks) * p[order_]
            bh_rev_cummin = np.minimum.accumulate(bh_vals[::-1])[::-1]
            adj = np.empty_like(p); adj[order_] = np.minimum(bh_rev_cummin, 1.0)
            out[mask] = adj; return out
        raise ValueError("p_correction must be 'BH'|'holm'|'bonferroni'|'none'")

    def _norm_key(col):
        if col.endswith("-baseline"): return col[:-9]
        if col.endswith("-prompt"):   return col[:-7]
        return col

    def _lighten(color, factor=0.55):
        r, g, b = to_rgb(color)
        return (1 - factor*(1-r), 1 - factor*(1-g), 1 - factor*(1-b))

    def _winsorize(arr, limits):
        arr = np.asarray(arr, float)
        if arr.size == 0: return arr
        lo, hi = np.nanpercentile(arr, [limits[0]*100, limits[1]*100])
        return np.clip(arr, lo, hi)

    role_suffix = {'Retailer':'r','Distributor':'d','Wholesaler':'w','Factory':'f'}
    role_params = {role: [f"theta_{s}", f"alpha_s_{s}", f"beta_{s}", f"s_prime_{s}"]
                   for role, s in role_suffix.items()}

    base_cols   = [c for c in df_means_base.columns   if c != human_dir]
    prompt_cols = [c for c in df_means_prompt.columns if c != human_dir]
    base_map    = {_norm_key(c): c for c in base_cols}
    prompt_map  = {_norm_key(c): c for c in prompt_cols}
    model_keys  = [k for k in base_map if k in prompt_map]
    if name_map is not None:
        allowed = set(name_map.keys())
        model_keys = [k for k in model_keys if k in allowed]
    if not model_keys:
        raise ValueError("No overlapping model keys.")
    if name_map is None:
        name_map = {k: k for k in model_keys}

    if order is not None:
        disp_to_keys = {}
        for k in model_keys:
            disp_to_keys.setdefault(name_map.get(k, k), []).append(k)
        collisions = {d: ks for d, ks in disp_to_keys.items() if len(ks) > 1}
        if collisions:
            raise ValueError("Duplicate display names in name_map:\n" +
                             "\n".join(f"  {d}: {ks}" for d, ks in collisions.items()))
        ordered, unknown = [], []
        for disp in order:
            if disp in disp_to_keys: ordered.append(disp_to_keys[disp][0])
            else: unknown.append(disp)
        if unknown: raise ValueError(f"Unknown display names in `order`: {unknown}")
        remaining = sorted([k for k in model_keys if k not in ordered],
                           key=lambda k: name_map.get(k, k))
        model_keys = ordered + remaining
    else:
        model_keys = sorted(model_keys, key=lambda k: name_map.get(k, k))

    human_stats = {}
    for role, rows in role_params.items():
        for p in rows:
            try: hr = df_all_samples_base.loc[(human_dir, p)].astype(float)
            except KeyError: hr = pd.Series(dtype=float)
            hr = hr[np.isfinite(hr)] if len(hr) else hr
            human_stats[p] = {
                'mean':   float(np.nanmean(hr.values))   if hr.size       else np.nan,
                'std':    float(np.nanstd(hr.values, ddof=1)) if hr.size > 1 else np.nan,
                'median': float(np.nanmedian(hr.values)) if hr.size       else np.nan,
                'iqr':    float(np.nanpercentile(hr.values, 75) -
                                np.nanpercentile(hr.values, 25)) if hr.size else np.nan,
            }

    def _aggregated(df_all, col_map, method):
        out = {k: [] for k in model_keys}; out['human'] = []
        for k in list(model_keys) + ['human']:
            col = col_map.get(k, k) if k != 'human' else human_dir
            normed = []
            for role, rows in role_params.items():
                for p in rows:
                    try:
                        s = df_all.loc[(col, p)].astype(float)
                        s = s[np.isfinite(s)]
                        h_mean, h_std = human_stats[p]['mean'], human_stats[p]['std']
                        if method == 'none':
                            sn = s / h_mean if np.isfinite(h_mean) and h_mean != 0 else pd.Series(dtype=float)
                        elif method == 'zscore':
                            sn = ((s - h_mean) / h_std) if (np.isfinite(h_mean) and np.isfinite(h_std) and h_std > 0) else pd.Series(dtype=float)
                        elif method == 'winsorize':
                            if np.isfinite(h_mean) and h_mean != 0:
                                sn = pd.Series(_winsorize(s.values, winsor_limits) / h_mean, index=s.index)
                            else:
                                sn = pd.Series(dtype=float)
                        elif method == 'median':
                            sn = s / h_mean if np.isfinite(h_mean) and h_mean != 0 else pd.Series(dtype=float)
                        else:
                            raise ValueError(f"Unknown robust_method: {method}")
                        normed.append(sn)
                    except KeyError:
                        pass
            if not normed: out[k] = np.array([]); continue
            common = normed[0].index
            for s in normed[1:]: common = common.intersection(s.index)
            if len(common) == 0:
                out[k] = np.array([])
            else:
                stacked = np.vstack([s.loc[common].values for s in normed])
                out[k] = np.nanmedian(stacked, axis=0) if method == 'median' else np.nanmean(stacked, axis=0)
        return out

    scores_base   = _aggregated(df_all_samples_base,   base_map,   robust_method)
    scores_prompt = _aggregated(df_all_samples_prompt, prompt_map, robust_method)
    human_scores  = scores_base['human']

    means_b, errs_b, means_p, errs_p, p_raw_b, p_raw_p = [], [], [], [], [], []
    for k in model_keys:
        xb, xp = scores_base[k], scores_prompt[k]
        means_b.append(np.nanmean(xb) if xb.size else np.nan)
        errs_b.append(_err_from(xb, err) if xb.size else np.nan)
        means_p.append(np.nanmean(xp) if xp.size else np.nan)
        errs_p.append(_err_from(xp, err) if xp.size else np.nan)
        p_raw_b.append(ttest_ind(xb, human_scores, equal_var=False, nan_policy='omit')[1]
                       if xb.size >= 2 and human_scores.size >= 2 else np.nan)
        p_raw_p.append(ttest_ind(xp, human_scores, equal_var=False, nan_policy='omit')[1]
                       if xp.size >= 2 and human_scores.size >= 2 else np.nan)
    p_adj_b = _adjust_pvals(np.array(p_raw_b), p_correction)
    p_adj_p = _adjust_pvals(np.array(p_raw_p), p_correction)

    # Figure: bars = mean across sessions (heuristic-aware ± s.e.m.), dots = sessions.
    fig, ax = plt.subplots(figsize=(HALF_WIDTH * 1.35, 62 * MM))
    set_dot_spacing(np.concatenate([np.asarray(scores[k], float) for scores in (scores_base, scores_prompt)
                                    for k in model_keys]), frac=0.018)
    for i, k in enumerate(model_keys):
        color = SET2_BARS[i % len(SET2_BARS)]
        bar_with_dots(ax, i - 0.2, scores_base[k], color, 0.38, baseline=True, err=None, dot_size=2.2)
        bar_with_dots(ax, i + 0.2, scores_prompt[k], color, 0.38, dot_size=2.2)
    ref = 0.0 if robust_method == 'zscore' else 1.0
    ref_label = "Human mean (z = 0)" if robust_method == 'zscore' else "Empirical benchmark"
    ax.axhline(ref, color="#555555", ls="--", lw=0.8, zorder=1)
    ax.set_xticks(range(len(model_keys)))
    ax.set_xticklabels([name_map.get(k, k) for k in model_keys], rotation=30, ha="right")
    ax.set_ylabel("Z-score (aggregated)" if robust_method == 'zscore' else "Normalized parameter value")
    ax.legend(handles=condition_handles([Line2D([], [], color="#555555", ls="--", lw=0.8, label=ref_label)]),
              loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)
    save_figure(fig, savepath)
    plt.show()

    if not return_table:
        return fig

    rows = [{
        'model_key': k, 'model_display': name_map.get(k, k),
        'mean_baseline': means_b[i], 'sem_baseline': _err_from(scores_base[k], 'sem'),
        'mean_prompt':   means_p[i], 'sem_prompt':   _err_from(scores_prompt[k], 'sem'),
        'p_raw_baseline_vs_human': p_raw_b[i], 'p_adj_baseline_vs_human': p_adj_b[i],
        'p_raw_prompt_vs_human':   p_raw_p[i], 'p_adj_prompt_vs_human':   p_adj_p[i],
    } for i, k in enumerate(model_keys)]
    return fig, pd.DataFrame(rows)


# --- Run aggregated overlay ---
fig_agg, tbl_agg = plot_overlay_aggregated(
    overall_means_by_dir,        all_samples_by_dir,
    overall_means_prompt_by_dir, all_samples_prompt_by_dir,
    human_dir='human',
    robust_method='median',
    name_map=OVERLAY_NAME_MAP,
    order=["GPT-OSS 20B", "Llama-3.1 8B", "GPT-OSS 120B", 'Mistral Small', "Kimi K2", "Llama-3.3 70B", "Qwen-3.6-35B-A3B"],
    savepath=PLOT_DIR / "main2.pdf",
    return_table=True,
)

In [ ]:
@nature_style
def plot_overlay_per_param(
    df_means_base, df_all_samples_base,
    df_means_prompt, df_all_samples_prompt,
    *,
    human_dir='human',
    err='sem',
    p_correction='BH',
    alpha=0.05,
    robust_method='zscore',          # 'none' | 'zscore' | 'winsorize' | 'median'
    winsor_limits=(0.05, 0.95),
    name_map=None,
    order=None,                      # list of *display names* for x-axis order
    savepath=None,
    return_table=True,
    layout='2x2',                    # '2x2' or '1x4'
):
    """
    Four-subplot bar plot: one panel per parameter type (theta, alpha_s, beta, s_prime).
    Within each panel, draws are normalized per-parameter (across the 4 roles for
    that parameter type) then aggregated across roles for each model.
    """

    # ---------------- helpers ----------------
    def _err_from(vals, mode):
        vals = np.asarray(vals, float); vals = vals[np.isfinite(vals)]
        n = vals.size
        if n == 0: return np.nan
        sd = np.nanstd(vals, ddof=1) if n > 1 else 0.0
        return {'std': sd, 'sem': sd/np.sqrt(n) if n else np.nan,
                'ci' : 1.96*sd/np.sqrt(n) if n else np.nan}[mode]

    def _adjust_pvals(pvals, method):
        pvals = np.asarray(pvals, float)
        out = np.full_like(pvals, np.nan, float)
        mask = np.isfinite(pvals); p = pvals[mask]; m = p.size
        if m == 0 or method.lower() == 'none':
            out[mask] = p; return out
        ml = method.lower()
        if ml == 'bonferroni': out[mask] = np.minimum(p * m, 1.0); return out
        if ml == 'holm':
            order_ = np.argsort(p); adj = np.empty_like(p)
            for i, idx in enumerate(order_): adj[idx] = (m - i) * p[idx]
            adj[order_] = np.maximum.accumulate(np.minimum(adj[order_], 1.0))
            out[mask] = np.minimum(adj, 1.0); return out
        if ml == 'bh':
            order_ = np.argsort(p); ranks = np.arange(1, m+1)
            bh_vals = (m / ranks) * p[order_]
            bh_rev_cummin = np.minimum.accumulate(bh_vals[::-1])[::-1]
            adj = np.empty_like(p); adj[order_] = np.minimum(bh_rev_cummin, 1.0)
            out[mask] = adj; return out
        raise ValueError("p_correction must be 'BH'|'holm'|'bonferroni'|'none'")

    def _stars(p):
        if not np.isfinite(p): return ""
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        return ""

    def _norm_key(col):
        if col.endswith("-baseline"): return col[:-9]
        if col.endswith("-prompt"):   return col[:-7]
        return col

    def _lighten(color, factor=0.55):
        r, g, b = to_rgb(color)
        return (1 - factor*(1-r), 1 - factor*(1-g), 1 - factor*(1-b))

    def _winsorize(arr, limits):
        arr = np.asarray(arr, float)
        if arr.size == 0: return arr
        lo, hi = np.nanpercentile(arr, [limits[0]*100, limits[1]*100])
        return np.clip(arr, lo, hi)

    # ---------------- parameter / role bookkeeping ----------------
    role_suffix  = {'Retailer':'r','Distributor':'d','Wholesaler':'w','Factory':'f'}
    PARAM_TYPES  = ['theta', 'alpha_s', 'beta', 's_prime']
    # Group across roles for each parameter type:
    #   theta   -> [theta_r, theta_d, theta_w, theta_f]
    #   alpha_s -> [alpha_s_r, alpha_s_d, alpha_s_w, alpha_s_f]   ... etc
    param_roles  = {p: [f"{p}_{s}" for s in role_suffix.values()] for p in PARAM_TYPES}
    all_params   = [p for pl in param_roles.values() for p in pl]

    # ---------------- model-key resolution ----------------
    base_cols   = [c for c in df_means_base.columns   if c != human_dir]
    prompt_cols = [c for c in df_means_prompt.columns if c != human_dir]
    base_map    = {_norm_key(c): c for c in base_cols}
    prompt_map  = {_norm_key(c): c for c in prompt_cols}
    model_keys  = [k for k in base_map if k in prompt_map]
    if name_map is not None:
        allowed = set(name_map.keys())
        model_keys = [k for k in model_keys if k in allowed]
    if not model_keys:
        raise ValueError("No overlapping model keys.")
    if name_map is None:
        name_map = {k: k for k in model_keys}

    if order is not None:
        disp_to_keys = {}
        for k in model_keys:
            disp_to_keys.setdefault(name_map.get(k, k), []).append(k)
        collisions = {d: ks for d, ks in disp_to_keys.items() if len(ks) > 1}
        if collisions:
            raise ValueError("Duplicate display names in name_map:\n" +
                             "\n".join(f"  {d}: {ks}" for d, ks in collisions.items()))
        ordered, unknown = [], []
        for disp in order:
            if disp in disp_to_keys: ordered.append(disp_to_keys[disp][0])
            else: unknown.append(disp)
        if unknown: raise ValueError(f"Unknown display names in `order`: {unknown}")
        remaining = sorted([k for k in model_keys if k not in ordered],
                           key=lambda k: name_map.get(k, k))
        model_keys = ordered + remaining
    else:
        model_keys = sorted(model_keys, key=lambda k: name_map.get(k, k))

    # ---------------- per-parameter human stats (computed once for all 16) ----------------
    human_stats = {}
    for p in all_params:
        try: hr = df_all_samples_base.loc[(human_dir, p)].astype(float)
        except KeyError: hr = pd.Series(dtype=float)
        hr = hr[np.isfinite(hr)] if len(hr) else hr
        human_stats[p] = {
            'mean':   float(np.nanmean(hr.values))   if hr.size       else np.nan,
            'std':    float(np.nanstd(hr.values, ddof=1)) if hr.size > 1 else np.nan,
            'median': float(np.nanmedian(hr.values)) if hr.size       else np.nan,
        }

    def _aggregated_for_params(df_all, col_map, param_list, method):
        """Same normalization as before, but only over `param_list`."""
        out = {k: [] for k in model_keys}; out['human'] = []
        for k in list(model_keys) + ['human']:
            col = col_map.get(k, k) if k != 'human' else human_dir
            normed = []
            for p in param_list:
                try:
                    s = df_all.loc[(col, p)].astype(float)
                    s = s[np.isfinite(s)]
                    h_mean, h_std = human_stats[p]['mean'], human_stats[p]['std']
                    if method == 'none':
                        sn = s / h_mean if np.isfinite(h_mean) and h_mean != 0 else pd.Series(dtype=float)
                    elif method == 'zscore':
                        sn = ((s - h_mean) / h_std) if (np.isfinite(h_mean) and np.isfinite(h_std) and h_std > 0) else pd.Series(dtype=float)
                    elif method == 'winsorize':
                        if np.isfinite(h_mean) and h_mean != 0:
                            sn = pd.Series(_winsorize(s.values, winsor_limits) / h_mean, index=s.index)
                        else:
                            sn = pd.Series(dtype=float)
                    elif method == 'median':
                        sn = s / h_mean if np.isfinite(h_mean) and h_mean != 0 else pd.Series(dtype=float)
                    else:
                        raise ValueError(f"Unknown robust_method: {method}")
                    normed.append(sn)
                except KeyError:
                    pass
            if not normed:
                out[k] = np.array([]); continue
            common = normed[0].index
            for s in normed[1:]: common = common.intersection(s.index)
            if len(common) == 0:
                out[k] = np.array([])
            else:
                stacked = np.vstack([s.loc[common].values for s in normed])
                out[k] = (np.nanmedian(stacked, axis=0) if method == 'median'
                          else np.nanmean(stacked, axis=0))
        return out

    # ---------------- figure layout ----------------
    # Bars = mean across sessions (heuristic-aware ± s.e.m.), dots = sessions;
    # stars = adjusted P for heuristic-aware vs the Sterman sessions (Welch t-test).
    if layout == '2x2':
        fig, axes = plt.subplots(2, 2, figsize=(FULL_WIDTH, 110 * MM))
        axes = axes.ravel()
    elif layout == '1x4':
        fig, axes = plt.subplots(1, 4, figsize=(FULL_WIDTH, 55 * MM), sharey=False)
    else:
        raise ValueError("layout must be '2x2' or '1x4'")

    table_rows = []

    # ---------------- one subplot per parameter type ----------------
    for ax_idx, param in enumerate(PARAM_TYPES):
        ax         = axes[ax_idx]
        param_list = param_roles[param]

        scores_base   = _aggregated_for_params(df_all_samples_base,   base_map,   param_list, robust_method)
        scores_prompt = _aggregated_for_params(df_all_samples_prompt, prompt_map, param_list, robust_method)
        human_scores  = scores_base['human']

        means_b, errs_b, means_p, errs_p, p_raw_b, p_raw_p = [], [], [], [], [], []
        for k in model_keys:
            xb, xp = scores_base[k], scores_prompt[k]
            means_b.append(np.nanmean(xb) if xb.size else np.nan)
            errs_b.append(_err_from(xb, err) if xb.size else np.nan)
            means_p.append(np.nanmean(xp) if xp.size else np.nan)
            errs_p.append(_err_from(xp, err) if xp.size else np.nan)
            p_raw_b.append(ttest_ind(xb, human_scores, equal_var=False, nan_policy='omit')[1]
                           if xb.size >= 2 and human_scores.size >= 2 else np.nan)
            p_raw_p.append(ttest_ind(xp, human_scores, equal_var=False, nan_policy='omit')[1]
                           if xp.size >= 2 and human_scores.size >= 2 else np.nan)
        p_adj_b = _adjust_pvals(np.array(p_raw_b), p_correction)
        p_adj_p = _adjust_pvals(np.array(p_raw_p), p_correction)

        # Bars and dots
        all_values = np.concatenate([np.asarray(scores[k], float) for k in model_keys
                                     for scores in (scores_base, scores_prompt)])
        set_dot_spacing(all_values, frac=0.018)
        star_y = np.nanmax(all_values) + (np.nanmax(all_values) - np.nanmin(all_values)) * 0.05
        for i, k in enumerate(model_keys):
            color = SET2_BARS[i % len(SET2_BARS)]
            bar_with_dots(ax, i - 0.2, scores_base[k], color, 0.38, baseline=True, err=None, dot_size=1.6)
            bar_with_dots(ax, i + 0.2, scores_prompt[k], color, 0.38, dot_size=1.6)
            lab = _stars(p_adj_p[i])
            if lab:
                ax.text(i + 0.2, star_y, lab, ha="center", va="bottom", fontsize=5.5)

        # Reference line
        ax.axhline(0.0 if robust_method == 'zscore' else 1.0, color="#555555", ls="--", lw=0.8, zorder=1)
        ax.set_ylabel("Z-score" if robust_method == 'zscore' else "Normalized value")
        ax.set_xticks(range(len(model_keys)))
        ax.set_xticklabels([name_map.get(k, k) for k in model_keys], rotation=30, ha="right")
        ax.set_title(PARAM_LABELS[param])

        # Collect table rows
        for i, k in enumerate(model_keys):
            table_rows.append({
                'parameter': param,
                'model_key': k, 'model_display': name_map.get(k, k),
                'mean_baseline': means_b[i], 'sem_baseline': _err_from(scores_base[k], 'sem'),
                'mean_prompt':   means_p[i], 'sem_prompt':   _err_from(scores_prompt[k], 'sem'),
                'p_raw_baseline_vs_human': p_raw_b[i], 'p_adj_baseline_vs_human': p_adj_b[i],
                'p_raw_prompt_vs_human':   p_raw_p[i], 'p_adj_prompt_vs_human':   p_adj_p[i],
            })

    # ---------------- shared legend ----------------
    ref_label = "Human mean (z = 0)" if robust_method == 'zscore' else "Empirical benchmark"
    fig.legend(handles=condition_handles([Line2D([], [], color="#555555", ls="--", lw=0.8, label=ref_label)]),
               loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=4)
    fig.tight_layout(h_pad=1.5, w_pad=1.5)
    save_figure(fig, savepath)
    plt.show()

    if not return_table:
        return fig
    return fig, pd.DataFrame(table_rows)


# --- Run per-parameter overlay ---
fig_pp, tbl_pp = plot_overlay_per_param(
    overall_means_by_dir,        all_samples_by_dir,
    overall_means_prompt_by_dir, all_samples_prompt_by_dir,
    human_dir='human',
    robust_method='median',
    name_map=OVERLAY_NAME_MAP,
    order=["GPT-OSS 20B", "Llama-3.1 8B", "GPT-OSS 120B", 'Mistral Small', "Kimi K2",
           "Llama-3.3 70B", "Qwen-3.6-35B-A3B", ],
    savepath=PLOT_DIR / "main2_per_param.pdf",
    return_table=True,
    layout='2x2',          # try '1x4' for a wide single-row layout
)

## 26. Table 7 Human Benchmarks (Low CR vs High CR)

A second set of human benchmarks from Sterman Table 7:
- **LOW_CR**: Low-Coordination-Ratio mean parameters
- **HIGH_CR**: High-Coordination-Ratio mean parameters


In [ ]:
LOW_CR = {
    "theta_r": 0.138, "alpha_s_r": 0.341, "beta_r": 0.054, "s_prime_r": 22,
    "theta_w": 0.253, "alpha_s_w": 0.324, "beta_w": 0.079, "s_prime_w": 17,
    "theta_d": 0.246, "alpha_s_d": 0.406, "beta_d": 0.038, "s_prime_d": 28,
    "theta_f": 0.389, "alpha_s_f": 0.543, "beta_f": 0.028, "s_prime_f": 21,
}
HIGH_CR = {
    "theta_r": 0.130, "alpha_s_r": 0.287, "beta_r": 0.238, "s_prime_r": 19,
    "theta_w": 0.366, "alpha_s_w": 0.255, "beta_w": 0.150, "s_prime_w": 21,
    "theta_d": 0.391, "alpha_s_d": 0.292, "beta_d": 0.203, "s_prime_d": 20,
    "theta_f": 0.672, "alpha_s_f": 0.412, "beta_f": 0.235, "s_prime_f": 13,
}

_ROWS_ORDER = [
    "theta_r","alpha_s_r","beta_r","s_prime_r",
    "theta_d","alpha_s_d","beta_d","s_prime_d",
    "theta_w","alpha_s_w","beta_w","s_prime_w",
    "theta_f","alpha_s_f","beta_f","s_prime_f",
]

human_df_table7   = pd.DataFrame({
    "sim_0": [LOW_CR[p]  for p in _ROWS_ORDER],
    "sim_1": [HIGH_CR[p] for p in _ROWS_ORDER],
}, index=_ROWS_ORDER, dtype=float)
human_mean_table7 = human_df_table7.mean(axis=1)


def append_table7_humans(all_samples_by_dir, overall_means_by_dir):
    """Replace the 'human' directory with the Table-7 (LOW/HIGH CR) two-sim version."""
    if isinstance(all_samples_by_dir.index, pd.MultiIndex):
        mask = all_samples_by_dir.index.get_level_values("directory") != "human"
        all_samples_by_dir = all_samples_by_dir[mask]
    human_stacked = human_df_table7.copy()
    human_stacked["directory"] = "human"
    human_stacked["parameter"] = human_stacked.index
    human_stacked = human_stacked.set_index(["directory", "parameter"])
    all_new   = pd.concat([all_samples_by_dir, human_stacked], axis=0)
    means_new = overall_means_by_dir.copy()
    if "human" in means_new.columns:
        means_new = means_new.drop(columns=["human"])
    means_new["human"] = human_mean_table7
    return all_new, means_new


all_samples_by_dir_t7,       overall_means_by_dir_t7       = append_table7_humans(
    all_samples_by_dir, overall_means_by_dir)
all_samples_prompt_by_dir_t7, overall_means_prompt_by_dir_t7 = append_table7_humans(
    all_samples_prompt_by_dir, overall_means_prompt_by_dir)


## 27. Aggregated Plot with Two Human Reference Lines (Low CR / High CR)


In [ ]:
T7_NAME_MAP = {
    "kimi": "Kimi K2", "qwen3": "Qwen-3.6-35B-A3B", "llama70b": "Llama-3.3 70B",
    "llama2": "Llama-2 7B", "oss20b": "GPT-OSS 20B", "llama3": "Llama-3.1 8B",
    "oss120b": "GPT-OSS 120B", 'mistral': "Mistral Small"
}
T7_ORDER = ["GPT-OSS 20B", "Llama-3.1 8B", "GPT-OSS 120B", 'Mistral Small',"Kimi K2", "Llama-2 7B", "Llama-3.3 70B", "Qwen-3.6-35B-A3B",]
T7_ROBUST = "median"
T7_HUMAN_DIR = "human"
T7_ERR = "sem"

role_suffix = {'Retailer':'r','Distributor':'d','Wholesaler':'w','Factory':'f'}
role_params = {role: [f"theta_{s}", f"alpha_s_{s}", f"beta_{s}", f"s_prime_{s}"]
               for role, s in role_suffix.items()}

def _t7_lighten(color, factor=0.55):
    r, g, b = to_rgb(color)
    return (1 - factor*(1-r), 1 - factor*(1-g), 1 - factor*(1-b))

def _t7_err_from(vals, mode):
    vals = np.asarray(vals, float); vals = vals[np.isfinite(vals)]
    n = vals.size
    if n == 0: return np.nan
    sd = np.nanstd(vals, ddof=1) if n > 1 else 0.0
    return {'std': sd, 'sem': sd/np.sqrt(n) if n else np.nan,
            'ci' : 1.96*sd/np.sqrt(n) if n else np.nan}[mode]

def _t7_norm_key(col):
    if col.endswith("-baseline"): return col[:-9]
    if col.endswith("-prompt"):   return col[:-7]
    return col

base_cols_t7   = [c for c in overall_means_by_dir_t7.columns        if c != T7_HUMAN_DIR]
prompt_cols_t7 = [c for c in overall_means_prompt_by_dir_t7.columns if c != T7_HUMAN_DIR]
base_map_t7    = {_t7_norm_key(c): c for c in base_cols_t7}
prompt_map_t7  = {_t7_norm_key(c): c for c in prompt_cols_t7}
model_keys_t7  = [k for k in base_map_t7 if k in prompt_map_t7 and k in T7_NAME_MAP]
disp_to_key    = {T7_NAME_MAP[k]: k for k in model_keys_t7}
model_keys_t7  = [disp_to_key[d] for d in T7_ORDER if d in disp_to_key]

human_stats_t7 = {}
for role, rows in role_params.items():
    for p in rows:
        hr = all_samples_by_dir_t7.loc[(T7_HUMAN_DIR, p)].astype(float)
        hr = hr[np.isfinite(hr)]
        human_stats_t7[p] = {'mean': float(np.nanmean(hr.values)) if hr.size else np.nan}

def _t7_aggregated(df_all, col_map, model_keys_list):
    out = {}
    for k in model_keys_list:
        col = col_map[k]
        normed = []
        for role, rows in role_params.items():
            for p in rows:
                try:
                    s = df_all.loc[(col, p)].astype(float)
                    s = s[np.isfinite(s)]
                    h_mean = human_stats_t7[p]['mean']
                    if np.isfinite(h_mean) and h_mean != 0:
                        normed.append(s / h_mean)
                except KeyError:
                    pass
        if not normed:
            out[k] = np.array([]); continue
        common = normed[0].index
        for s in normed[1:]: common = common.intersection(s.index)
        if len(common) == 0:
            out[k] = np.array([])
        else:
            stacked = np.vstack([s.loc[common].values for s in normed])
            out[k] = (np.nanmedian(stacked, axis=0) if T7_ROBUST == 'median'
                      else np.nanmean(stacked, axis=0))
    return out

scores_base_t7   = _t7_aggregated(all_samples_by_dir_t7,        base_map_t7,   model_keys_t7)
scores_prompt_t7 = _t7_aggregated(all_samples_prompt_by_dir_t7, prompt_map_t7, model_keys_t7)

def _human_normalised_score(cr_dict):
    vals = []
    for role, rows in role_params.items():
        for p in rows:
            h_mean = human_stats_t7[p]['mean']
            if np.isfinite(h_mean) and h_mean != 0:
                vals.append(cr_dict[p] / h_mean)
    return float(np.median(vals)) if T7_ROBUST == 'median' else float(np.mean(vals))

human_low_cr_score  = _human_normalised_score(LOW_CR)
human_high_cr_score = _human_normalised_score(HIGH_CR)

means_b_t7, errs_b_t7, means_p_t7, errs_p_t7 = [], [], [], []
for k in model_keys_t7:
    xb = scores_base_t7[k]; xp = scores_prompt_t7[k]
    means_b_t7.append(np.nanmean(xb) if xb.size else np.nan)
    errs_b_t7.append(_t7_err_from(xb, T7_ERR) if xb.size else np.nan)
    means_p_t7.append(np.nanmean(xp) if xp.size else np.nan)
    errs_p_t7.append(_t7_err_from(xp, T7_ERR) if xp.size else np.nan)

# Figure: bars = mean across sessions (heuristic-aware ± s.e.m.), dots = sessions;
# dashed lines = the low- and high-CR human medians, normalized the same way.
@nature_style
def plot_aggregated_vs_cr_medians(savepath=None):
    fig, ax = plt.subplots(figsize=(HALF_WIDTH * 1.35, 62 * MM))
    set_dot_spacing(np.concatenate([np.asarray(scores[k], float) for scores in (scores_base_t7, scores_prompt_t7)
                                    for k in model_keys_t7]), frac=0.018)
    for i, k in enumerate(model_keys_t7):
        color = SET2_BARS[i % len(SET2_BARS)]
        bar_with_dots(ax, i - 0.2, scores_base_t7[k], color, 0.38, baseline=True, err=None, dot_size=2.2)
        bar_with_dots(ax, i + 0.2, scores_prompt_t7[k], color, 0.38, dot_size=2.2)
    reference_lines = [("Median human (low CR)", human_low_cr_score, CR_COLORS["low CR"]),
                       ("Median human (high CR)", human_high_cr_score, CR_COLORS["high CR"])]
    for _, y, color in reference_lines:
        ax.axhline(y, color=color, ls="--", lw=0.8, zorder=1)
    ax.set_xticks(range(len(model_keys_t7)))
    ax.set_xticklabels([T7_NAME_MAP[k] for k in model_keys_t7], rotation=30, ha="right")
    ax.set_ylabel("Normalized parameter value")
    ax.legend(handles=condition_handles([Line2D([], [], color=color, ls="--", lw=0.8, label=label)
                                         for label, _, color in reference_lines]),
              loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)
    save_figure(fig, savepath)
    plt.show()


plot_aggregated_vs_cr_medians(PLOT_DIR / "parameter_overlay_aggregated_table7.pdf")

## 29. Sterman → Posterior Helpers (parse / normalize / summarize)

Helpers used by both the per-Sterman-session pipeline and the per-LLM-directory
pipeline. Canonical order: 4 roles × 4 params = 16 columns.


In [ ]:
ROLE_FROM_SUFFIX = {"r": "Retailer", "w": "Wholesaler", "d": "Distributor", "f": "Factory"}
ROLE_ORDER       = ["Retailer", "Wholesaler", "Distributor", "Factory"]
PARAM_ORDER      = ["theta", "alpha_s", "beta", "s_prime"]
CANON_COLS       = [f"{r}_{p}" for r in ROLE_ORDER for p in PARAM_ORDER]


def _sterman_session_to_load_kwargs(sess):
    """Convert a single _STERMAN_11 entry into kwargs for load_data()."""
    return dict(
        theta_r=sess["Retailer"]["theta"],   alpha_s_r=sess["Retailer"]["alpha_s"],   beta_r=sess["Retailer"]["beta"],   s_prime_r=sess["Retailer"]["s_prime"],
        theta_d=sess["Distributor"]["theta"],alpha_s_d=sess["Distributor"]["alpha_s"],beta_d=sess["Distributor"]["beta"],s_prime_d=sess["Distributor"]["s_prime"],
        theta_w=sess["Wholesaler"]["theta"], alpha_s_w=sess["Wholesaler"]["alpha_s"], beta_w=sess["Wholesaler"]["beta"], s_prime_w=sess["Wholesaler"]["s_prime"],
        theta_f=sess["Factory"]["theta"],    alpha_s_f=sess["Factory"]["alpha_s"],    beta_f=sess["Factory"]["beta"],    s_prime_f=sess["Factory"]["s_prime"],
    )


def _parse_param_and_role(col):
    """Accepts tuple, 'Retailer_theta', or 'theta_r' etc. Returns (role, param)."""
    if isinstance(col, tuple) and len(col) >= 2:
        return str(col[0]).strip(), str(col[1]).strip()
    name = str(col)
    parts = name.split("_")
    if parts and parts[0] in {"Retailer","Wholesaler","Distributor","Factory"}:
        role  = parts[0]
        param = "_".join(parts[1:]) if len(parts) > 1 else ""
        return role, param
    if len(parts) >= 3 and parts[-1] in ROLE_FROM_SUFFIX:
        return ROLE_FROM_SUFFIX[parts[-1]], "_".join(parts[:-1])
    if len(parts) == 2 and parts[-1] in ROLE_FROM_SUFFIX:
        return ROLE_FROM_SUFFIX[parts[-1]], parts[0]
    return "Unknown", name


def _normalize_param_columns(df_samples):
    """Returns (df_norm, ordered_cols) with standardized 'Role_param' column names."""
    id_cols = [c for c in df_samples.columns
               if c in ("directory","sim_idx","session_id","draw_id")]
    col_map = {}
    for c in df_samples.columns:
        if c in id_cols: continue
        role, param = _parse_param_and_role(c)
        col_map[c] = f"{role}_{param}"
    df_norm = df_samples.rename(columns=col_map).copy()
    ordered_cols = [c for c in CANON_COLS if c in df_norm.columns]
    return df_norm, ordered_cols


def _summarize_posterior(df_norm):
    """Tidy per-(role, param) posterior summary: mean/std/p05/p50/p95/n."""
    rows = []
    for c in df_norm.columns:
        if c in ("directory","sim_idx","session_id","draw_id"):
            continue
        s = pd.to_numeric(df_norm[c], errors="coerce").dropna()
        if s.empty: continue
        role, param = _parse_param_and_role(c)
        rows.append({
            "role": role, "param": param,
            "mean": float(s.mean()),
            "std":  float(s.std(ddof=1)) if len(s) > 1 else 0.0,
            "p05":  float(s.quantile(0.05)),
            "p50":  float(s.quantile(0.50)),
            "p95":  float(s.quantile(0.95)),
            "n":    int(len(s)),
        })
    out = pd.DataFrame(rows)
    if out.empty: return out
    out["role"]  = pd.Categorical(out["role"],  categories=ROLE_ORDER + ["Unknown"], ordered=True)
    out["param"] = pd.Categorical(out["param"], categories=PARAM_ORDER, ordered=True)
    return out.sort_values(["role","param"]).reset_index(drop=True)


## 30. Run Inference Over the 11 Sterman Sessions (Human Posteriors)


In [ ]:
def infer_over_sterman_sessions_all(sterman_list, approximator, workflow,
                                    num_samples=2000, seed=42):
    """
    For each of the 11 Sterman sessions:
      - Re-run the simulator with that session's parameters (via load_data).
      - Draw `num_samples` posterior samples from the approximator.
      - Normalize column names + collect tidy & wide DataFrames.
    """
    rng = np.random.default_rng(seed)
    all_summaries, all_samples_wide = [], []

    for session_id, sess in enumerate(sterman_list):
        kwargs = _sterman_session_to_load_kwargs(sess)
        obs_cases = load_data(**kwargs)
        try:
            samples = approximator.sample(
                conditions=obs_cases, num_samples=num_samples,
                seed=int(rng.integers(0, 2**31-1)))
        except TypeError:
            np.random.seed(int(rng.integers(0, 2**31-1)))
            samples = approximator.sample(conditions=obs_cases, num_samples=num_samples)

        df_raw = workflow.samples_to_data_frame(samples).copy()
        df_raw["session_id"] = session_id

        df_norm, ordered_cols = _normalize_param_columns(df_raw)
        summ = _summarize_posterior(df_norm)
        summ.insert(0, "session_id", session_id)
        all_summaries.append(summ)

        cols_to_keep = ordered_cols + ["session_id"]
        all_samples_wide.append(df_norm[cols_to_keep])

    summaries    = pd.concat(all_summaries, ignore_index=True)
    samples_wide = pd.concat(all_samples_wide, ignore_index=True)
    samples_wide["draw_id"] = samples_wide.groupby("session_id").cumcount()

    value_cols = [c for c in samples_wide.columns if c not in ("session_id","draw_id")]
    samples_long = (
        samples_wide
        .melt(id_vars=["session_id","draw_id"], value_vars=value_cols,
              var_name="role_param", value_name="value")
        .assign(role=lambda d: d["role_param"].str.split("_").str[0],
                param=lambda d: d["role_param"].str.split("_").str[1:].str.join("_"))
        [["session_id","draw_id","role","param","value"]]
        .sort_values(["session_id","draw_id","role","param"], ignore_index=True)
    )
    return summaries, samples_long, samples_wide


# Human posteriors (re-used by all subsequent PCA / distance cells)
human_summaries, human_samples_long, human_samples_wide = infer_over_sterman_sessions_all(
    _STERMAN_11, approximator, workflow, num_samples=10_000, seed=123)
display(human_summaries.head())
display(human_samples_long.head())
display(human_samples_wide.head())


## 31. Compute Posterior Samples for a List of Model Directories


In [ ]:
def compute_samples_for_dirs(dirs, sim_indices=range(10), *,
                             num_samples=10_000, verbose=True):
    """
    Like `compute_means_for_dirs` but keeps every posterior draw.
    Returns:
      samples_long : tidy (directory, sim_idx, draw_id, role, param, value)
      samples_wide : wide with 16 'Role_param' columns + (directory, sim_idx, draw_id)
      summaries    : per-(directory, sim_idx, role, param) stats
      mean_df      : mean across sims per directory (role/param index)
    """
    all_wide, all_summaries = [], []
    per_dir_param_means = {}

    for d in dirs:
        label = Path(d).name
        if verbose: print(f"\n=== Processing directory: {label} ===")

        per_sim_wide, per_sim_means, sim_names = [], [], []

        for i in sim_indices:
            sim_name = f"sim_{i}"
            if verbose: print(f"Processing {label}/{sim_name}...")
            try:
                obs_cases = load_simulated_data(str(d), sim_name=sim_name)
                samples   = approximator.sample(conditions=obs_cases, num_samples=num_samples)
                df_raw    = workflow.samples_to_data_frame(samples).copy()
                df_raw["directory"] = label
                df_raw["sim_idx"]   = i
                df_raw = df_raw.reset_index(drop=True)
                df_raw["draw_id"] = df_raw.index

                df_norm, ordered_cols = _normalize_param_columns(df_raw)
                keep_cols = ["directory","sim_idx","draw_id"] + ordered_cols
                per_sim_wide.append(df_norm[keep_cols])
                per_sim_means.append(df_norm[ordered_cols].mean(axis=0))
                sim_names.append(sim_name)

                sim_summary = _summarize_posterior(df_norm[keep_cols])
                if not sim_summary.empty:
                    sim_summary.insert(0, "sim_idx", i)
                    sim_summary.insert(0, "directory", label)
                    all_summaries.append(sim_summary)
            except (FileNotFoundError, ValueError) as e:
                if verbose: print(f"[{label}] Skipping {sim_name}: {e}")
            except Exception as e:
                if verbose: print(f"[{label}] Error in {sim_name}: {e}")

        if per_sim_wide:
            all_wide.append(pd.concat(per_sim_wide, ignore_index=True))
            df_sims = pd.concat(per_sim_means, axis=1)
            df_sims.columns = sim_names
            per_dir_param_means[label] = df_sims.mean(axis=1)
        elif verbose:
            print(f"[{label}] No simulation data was successfully processed.")

    if not all_wide:
        if verbose: print("\nNo directories produced results.")
        empty = pd.DataFrame()
        return empty, empty, empty, empty

    samples_wide = pd.concat(all_wide, ignore_index=True)
    value_cols = [c for c in samples_wide.columns
                  if c not in ("directory","sim_idx","draw_id")]
    samples_long = (
        samples_wide
        .melt(id_vars=["directory","sim_idx","draw_id"], value_vars=value_cols,
              var_name="role_param", value_name="value")
        .assign(role=lambda d: d["role_param"].str.split("_").str[0],
                param=lambda d: d["role_param"].str.split("_").str[1:].str.join("_"))
        [["directory","sim_idx","draw_id","role","param","value"]]
        .sort_values(["directory","sim_idx","draw_id","role","param"], ignore_index=True)
    )
    summaries = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()

    mean_df = (pd.DataFrame(per_dir_param_means)
                 .rename_axis("role_param").reset_index()
                 .assign(role=lambda d: d["role_param"].str.split("_").str[0],
                         param=lambda d: d["role_param"].str.split("_").str[1:].str.join("_"))
                 .set_index(["role","param"])
                 .drop(columns=["role_param"]).sort_index())

    if verbose:
        print("\nConstructed outputs:")
        print(f"- samples_wide: {samples_wide.shape}")
        print(f"- samples_long: {samples_long.shape}")
        print(f"- summaries:    {summaries.shape}")
        print(f"- mean_df:      {mean_df.shape}")

    return samples_long, samples_wide, summaries, mean_df


## 32. PCA / UMAP Embedding Helpers


In [ ]:
def _common_param_cols(df1, df2):
    c1 = [c for c in df1.columns if c in CANON_COLS]
    c2 = [c for c in df2.columns if c in CANON_COLS]
    return [c for c in c1 if c in c2]


def _subsample_per_group(df, group_col, n_per_group, random_state=123):
    return (df.groupby(group_col, group_keys=False)
              .apply(lambda d: d.sample(min(n_per_group, len(d)),
                                        random_state=random_state)))


def _build_combined(human_wide, model_wide, mode_rename=None):
    """Stack human + model samples with consistent 'source' / 'group' / 'mode' columns."""
    mode_rename = mode_rename or {}
    param_cols  = _common_param_cols(human_wide, model_wide)

    h = human_wide.copy()
    h["source"] = "Human"
    h["group"]  = h.get("session_id", 0).astype(str)
    h["mode"]   = "human"

    m = model_wide.copy()
    m["source"] = "Model"
    if "directory" in m and "sim_idx" in m:
        m["group"] = m["directory"].astype(str) + "#" + m["sim_idx"].astype(str)
    else:
        m["group"] = "ModelGroup"
    m["mode"] = m.get("directory", "model").astype(str).replace(mode_rename)

    keep = ["source","group","mode"] + (["draw_id"] if "draw_id" in h and "draw_id" in m else []) + param_cols
    return pd.concat([h[keep], m[keep]], ignore_index=True), param_cols


def _filter_outliers(df, param_cols, z_thresh=2.0):
    Z = np.abs(zscore(df[param_cols], axis=0, nan_policy="omit"))
    return df.loc[(Z < z_thresh).all(axis=1)].reset_index(drop=True)


def _embed_pca(df, param_cols, standardize=True):
    X = df[param_cols].values
    if standardize: X = StandardScaler().fit_transform(X)
    coords = PCA(n_components=2).fit_transform(X)
    return df.assign(PC1=coords[:,0], PC2=coords[:,1])


def _maybe_umap(df, param_cols, standardize=True,
                n_neighbors=30, min_dist=0.1, random_state=42):
    try:
        import umap
    except Exception:
        return None
    X = df[param_cols].values
    if standardize: X = StandardScaler().fit_transform(X)
    coords = umap.UMAP(n_components=2, n_neighbors=n_neighbors,
                       min_dist=min_dist, random_state=random_state).fit_transform(X)
    return df.assign(UMAP1=coords[:,0], UMAP2=coords[:,1])


def plot_embedding(human_wide, model_wide,
                   *, view="source", z_thresh=2.0,
                   subsample_on=None, n_per_group=4000,
                   do_umap=False, mode_rename=None, title_suffix=""):
    """PCA scatter (and optional UMAP) coloured by `view` in {'source','mode','group'}."""
    combined, param_cols = _build_combined(human_wide, model_wide, mode_rename)
    filtered = _filter_outliers(combined, param_cols, z_thresh=z_thresh)
    on = subsample_on or view
    filtered_sub = _subsample_per_group(filtered, on, n_per_group) if on else filtered

    pca_df = _embed_pca(filtered_sub, param_cols)
    plt.figure(figsize=(10, 8))
    for key, d in pca_df.groupby(view):
        plt.scatter(d["PC1"], d["PC2"], s=8, alpha=0.5, label=str(key))
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.title(f"PCA of Posterior Samples by {view.upper()} {title_suffix}".strip())
    plt.legend(loc="best", fontsize=8, markerscale=1.5)
    plt.grid(True); plt.show()

    if do_umap:
        umap_df = _maybe_umap(filtered_sub, param_cols)
        if umap_df is not None:
            plt.figure(figsize=(10, 8))
            for key, d in umap_df.groupby(view):
                plt.scatter(d["UMAP1"], d["UMAP2"], s=8, alpha=0.5, label=str(key))
            plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
            plt.title(f"UMAP of Posterior Samples by {view.upper()} {title_suffix}".strip())
            plt.legend(loc="best", fontsize=8, markerscale=1.5)
            plt.grid(True); plt.show()
        else:
            print("UMAP not available; skipped.")


## 33. Mode Means and Pairwise Distance Matrix


In [ ]:
def compute_mode_means(human_wide, model_wide, *, standardize=True, mode_rename=None):
    """
    Posterior mean vector per 'mode' (human, then each model dir, optionally renamed).
    If standardize=True, columns are z-scored across modes BEFORE distance computation.
    """
    mode_rename = mode_rename or {}
    param_cols  = _common_param_cols(human_wide, model_wide)

    h = human_wide.copy(); h["mode"] = "human"
    m = model_wide.copy()
    m["mode"] = m.get("directory", "model").astype(str).replace(mode_rename)

    combined = pd.concat(
        [h[["mode"]+param_cols], m[["mode"]+param_cols]], ignore_index=True)
    mode_means = combined.groupby("mode", sort=False)[param_cols].mean()

    if standardize:
        X = StandardScaler().fit_transform(mode_means.values)
        mode_means = pd.DataFrame(X, index=mode_means.index, columns=mode_means.columns)
    return mode_means


def compute_mean_distance_matrix(mode_means):
    """Pairwise Euclidean distance between each pair of mode-mean vectors."""
    dist = pairwise_distances(mode_means.values, metric="euclidean")
    return pd.DataFrame(dist, index=mode_means.index, columns=mode_means.index)


## 34. PCA + Distance Panel Plot (parameterized)


In [ ]:
@nature_style
def plot_pca_and_distance_panels(
    human_samples_wide,
    model_samples_wide,
    *,
    view="mode",                              # 'mode' | 'source' | 'group'
    z_thresh=1.5,                             # |z| < z_thresh kept
    subsample_on=None,
    n_per_group=1000,
    pca_standardize=True,
    means_standardize=True,
    mode_rename=None,
    cmap="viridis",                           # heatmap colormap
    scatter_palette="tab10",                  # str | list | dict
    savepath=None,                            # file path for figure
    force_human_palette_index=6,
    fontweight="bold",                        # text weight for (a)/(b) labels
    special_human_marker=False,               # X marker + grey for human in scatter
    rename_human_in_heatmap=False,            # rename 'human' -> 'Empirical benchmark' in heatmap
):
    """
    Two-panel figure:
      (a) PCA scatter of posterior draws coloured by `view`
      (b) Heatmap of pairwise Euclidean distances between per-mode posterior means
    """
    mode_rename = mode_rename or {}

    param_cols = _common_param_cols(human_samples_wide, model_samples_wide)
    if not param_cols:
        raise ValueError("No overlapping parameter columns between human and model frames.")

    h = human_samples_wide.copy()
    h["source"] = "Human"
    h["group"]  = h.get("session_id", 0).astype(str)
    h["mode"]   = "human"
    h = h[["source","group","mode","draw_id"] + param_cols]

    m = model_samples_wide.copy()
    m["source"] = "Model"
    m["group"]  = (m.get("directory","dir").astype(str) + "#" + m.get("sim_idx",0).astype(str)
                   if "directory" in m and "sim_idx" in m else "ModelGroup")
    m["mode"]   = m.get("directory","model").astype(str).replace(mode_rename)
    m = m[["source","group","mode","draw_id"] + param_cols]

    combined = pd.concat([h, m], ignore_index=True)

    Z = np.abs(zscore(combined[param_cols], axis=0, nan_policy="omit"))
    keep_mask = (Z < z_thresh).all(axis=1)
    filtered  = combined.loc[keep_mask].reset_index(drop=True)

    on = subsample_on or view
    if on is not None:
        parts = [d.sample(min(n_per_group, len(d)), random_state=123)
                 for _, d in filtered.groupby(on, dropna=False)]
        sample_for_pca = pd.concat(parts, ignore_index=True) if parts else filtered
    else:
        sample_for_pca = filtered

    X = sample_for_pca[param_cols].values
    if pca_standardize: X = StandardScaler().fit_transform(X)
    coords = PCA(n_components=2).fit_transform(X)
    sample_for_pca = sample_for_pca.assign(PC1=coords[:,0], PC2=coords[:,1])

    mode_means   = compute_mode_means(human_samples_wide, model_samples_wide,
                                      standardize=means_standardize, mode_rename=mode_rename)
    mean_dist_df = compute_mean_distance_matrix(mode_means)

    unique_groups = sample_for_pca[view].unique()
    if isinstance(scatter_palette, str):
        palette = sns.color_palette(scatter_palette, n_colors=len(unique_groups))
        color_map = dict(zip(unique_groups, palette))
        if force_human_palette_index is not None and "human" in unique_groups:
            extended = sns.color_palette(
                scatter_palette, n_colors=max(force_human_palette_index + 1, len(unique_groups)))
            color_map["human"] = extended[force_human_palette_index]
    elif isinstance(scatter_palette, list):
        if len(scatter_palette) < len(unique_groups):
            raise ValueError("Not enough colors provided in scatter_palette list.")
        color_map = dict(zip(unique_groups, scatter_palette))
        if force_human_palette_index is not None and "human" in unique_groups \
                and len(scatter_palette) > force_human_palette_index:
            color_map["human"] = scatter_palette[force_human_palette_index]
    elif isinstance(scatter_palette, dict):
        color_map = scatter_palette
    else:
        raise ValueError("scatter_palette must be str, list, or dict.")

    # Figure: (a) posterior draws in PCA space (benchmark draws as crosses);
    # (b) distances between posterior means, annotated.
    fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(FULL_WIDTH, 78 * MM), gridspec_kw={"width_ratios": [1.05, 1]})
    for key, d in sample_for_pca.groupby(view, sort=False):
        if special_human_marker and key == "human":
            ax_a.scatter(d["PC1"], d["PC2"], s=6, marker="x", color=BENCHMARK_X, linewidths=0.4,
                         alpha=0.6, label="Empirical benchmark", zorder=3)
        else:
            ax_a.scatter(d["PC1"], d["PC2"], s=2, color=BENCHMARK_X if key == "human" else color_map.get(key, "gray"),
                         edgecolors="none", alpha=0.5, label=str(key), zorder=2)
    ax_a.set_xlabel("PC1")
    ax_a.set_ylabel("PC2")
    leg = ax_a.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), markerscale=2.5, handletextpad=0.2)
    for lh in getattr(leg, "legend_handles", getattr(leg, "legendHandles", [])):
        lh.set_alpha(1.0)
    ax_a.text(-0.12, 1.04, "a", transform=ax_a.transAxes, fontsize=8, fontweight=fontweight)

    heat_df = (mean_dist_df.rename(index={"human": "Empirical benchmark"},
                                   columns={"human": "Empirical benchmark"})
               if rename_human_in_heatmap else mean_dist_df)
    im = ax_b.imshow(heat_df.values, cmap=cmap)
    for (r, c), v in np.ndenumerate(heat_df.values):
        ax_b.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=4.5,
                  color="white" if v > heat_df.values.max() * 0.6 else "black")
    ax_b.set_xticks(range(len(heat_df.columns)))
    ax_b.set_xticklabels(heat_df.columns, rotation=45, ha="right")
    ax_b.set_yticks(range(len(heat_df.index)))
    ax_b.set_yticklabels(heat_df.index)
    ax_b.spines[["top", "right"]].set_visible(True)
    cb = fig.colorbar(im, ax=ax_b, shrink=0.8, pad=0.02)
    cb.set_label("Euclidean distance")
    cb.outline.set_linewidth(0.5)
    ax_b.text(-0.55, 1.04, "b", transform=ax_b.transAxes, fontsize=8, fontweight=fontweight)

    fig.tight_layout(w_pad=2.0)
    save_figure(fig, savepath)
    plt.show()

    return {
        "pca_df": sample_for_pca[["source","group","mode","draw_id","PC1","PC2"]],
        "mean_dist_df": mean_dist_df,
        "mode_means": mode_means,
        "kept_samples": int(keep_mask.sum()),
        "total_samples": int(len(combined)),
    }


def _pretty_map_for_treatment(treatment):
    """Build {'<model>-<treatment>' -> 'Pretty Name'} (used as `mode_rename`)."""
    return {f"{k}-{treatment}": v for k, v in DEFAULT_MODEL_NAMES.items()}

## 35. Per-Experiment PCA & Distance Plots


In [ ]:
# -------- BASELINE --------
_, baseline_model_samples_wide, _, _ = compute_samples_for_dirs(
    DIRS_BASELINE, sim_indices=range(11),
    num_samples=num_samples, verbose=False)

_ = plot_pca_and_distance_panels(
    human_samples_wide, baseline_model_samples_wide,
    scatter_palette=PCA_COLORS, cmap="viridis_r",
    mode_rename=_pretty_map_for_treatment("baseline"),
    savepath=PLOT_DIR / "parameter_distribution_baseline.pdf",
    special_human_marker=True,
    rename_human_in_heatmap=True,
)

# -------- PROMPT --------
_, prompt_model_samples_wide, _, _ = compute_samples_for_dirs(
    DIRS_PROMPT, sim_indices=range(11),
    num_samples=num_samples, verbose=False)

_ = plot_pca_and_distance_panels(
    human_samples_wide, prompt_model_samples_wide,
    scatter_palette=PCA_COLORS, cmap="viridis_r",
    mode_rename=_pretty_map_for_treatment("prompt"),
    savepath=PLOT_DIR / "parameter_distribution_prompt.pdf",
    special_human_marker=True,
    rename_human_in_heatmap=True,
)

# -------- HISTORY (few-shot) --------
_, history_model_samples_wide, _, _ = compute_samples_for_dirs(
    DIRS_HISTORY, sim_indices=range(11),
    num_samples=num_samples, verbose=False)

_ = plot_pca_and_distance_panels(
    human_samples_wide, history_model_samples_wide,
    scatter_palette=PCA_COLORS, cmap="viridis_r",
    mode_rename=_pretty_map_for_treatment("alt"),
    savepath=PLOT_DIR / "parameter_distribution_alt.pdf",
    special_human_marker=True,
    rename_human_in_heatmap=True,
)

## 36. Optional: "Main viz" variant with X-marker for human

Variant of the PCA+distance panel used by the 'Main viz' section in the
original notebook: the human cluster is plotted as a grey X-marker overlay
and is renamed to 'Empirical Benchmark' in both the legend and the heatmap.
Saves to a separate file so it doesn't overwrite the standard panels above.


In [ ]:
_ = plot_pca_and_distance_panels(
    human_samples_wide, prompt_model_samples_wide,
    scatter_palette=PCA_COLORS,
    cmap="viridis_r",
    mode_rename=_pretty_map_for_treatment("prompt"),
    special_human_marker=True,
    rename_human_in_heatmap=True,
    savepath=PLOT_DIR / "main3.pdf",
)